In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load metric.csv
metric_df = pd.read_csv('metric.csv')
if 'timestamp' in metric_df.columns:
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s').dt.tz_localize(tz)
# Aggregate unique (cmdb_id, kpi_name) pairs with counts
metric_pairs = (
    metric_df.groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)
metric_top20 = metric_pairs.head(20)
metric_distinct_counts = pd.DataFrame({
    'distinct_cmdb_id': [metric_df['cmdb_id'].nunique()],
    'distinct_kpi_name': [metric_df['kpi_name'].nunique()]
})

# Load trace.csv
trace_df = pd.read_csv('trace.csv')
if 'timestamp' in trace_df.columns:
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s').dt.tz_localize(tz)
trace_pairs = (
    trace_df.groupby(['cmdb_id', 'trace_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)
trace_top20 = trace_pairs.head(20)
trace_distinct_counts = pd.DataFrame({
    'distinct_cmdb_id': [trace_df['cmdb_id'].nunique()],
    'distinct_trace_name': [trace_df['trace_name'].nunique()]
})

# Load log.csv
log_df = pd.read_csv('log.csv')
if 'timestamp' in log_df.columns:
    log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s').dt.tz_localize(tz)
log_pairs = (
    log_df.groupby(['cmdb_id', 'log_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)
log_top20 = log_pairs.head(20)
log_distinct_counts = pd.DataFrame({
    'distinct_cmdb_id': [log_df['cmdb_id'].nunique()],
    'distinct_log_name': [log_df['log_name'].nunique()]
})

# Load error_logs.csv
errors_df = pd.read_csv('error_logs.csv')
if 'timestamp' in errors_df.columns:
    errors_df['timestamp'] = pd.to_datetime(errors_df['timestamp'], unit='s').dt.tz_localize(tz)
error_total_count = len(errors_df)
error_counts_by_cmdb = (
    errors_df.groupby('cmdb_id', dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)
error_top20_by_cmdb = error_counts_by_cmdb.head(20)

# Compact outputs (only aggregates as requested)
metric_top20, metric_distinct_counts, trace_top20, trace_distinct_counts, log_top20, log_distinct_counts, error_total_count, error_top20_by_cmdb

```
Out[1]:
```
# Compose a plain-English summary using the previously loaded aggregates
metric_cmdbs = int(metric_distinct_counts.at[0, 'distinct_cmdb_id'])
metric_kpis = int(metric_distinct_counts.at[0, 'distinct_kpi_name'])
metric_top_examples = [
    f"{r['cmdb_id']} / {r['kpi_name']} (count={int(r['count'])})"
    for _, r in metric_top20.head(5).iterrows()
]

trace_cmdbs = int(trace_distinct_counts.at[0, 'distinct_cmdb_id'])
trace_names = int(trace_distinct_counts.at[0, 'distinct_trace_name'])
trace_top_examples = [
    f"{r['cmdb_id']} / {r['trace_name']} (count={int(r['count'])})"
    for _, r in trace_top20.head(5).iterrows()
]

log_cmdbs = int(log_distinct_counts.at[0, 'distinct_cmdb_id'])
log_names = int(log_distinct_counts.at[0, 'distinct_log_name'])
log_top_examples = [
    f"{r['cmdb_id']} / {r['log_name']} (count={int(r['count'])})"
    for _, r in log_top20.head(5).iterrows()
]

error_total = int(error_total_count)
error_top_examples = [
    f"{r['cmdb_id']} (count={int(r['count'])})"
    for _, r in error_top20_by_cmdb.head(5).iterrows()
] if not error_top20_by_cmdb.empty else []

summary = (
    "Summary of telemetry files:\n\n"
    f"metric.csv:\n"
    f" - distinct cmdb_id: {metric_cmdbs}\n"
    f" - distinct kpi_name: {metric_kpis}\n"
    f" - top pairs by record count (examples, up to 5):\n    - " + "\n    - ".join(metric_top_examples) + "\n\n"
    f"trace.csv:\n"
    f" - distinct cmdb_id: {trace_cmdbs}\n"
    f" - distinct trace_name: {trace_names}\n"
    f" - top pairs by record count (examples, up to 5):\n    - " + "\n    - ".join(trace_top_examples) + "\n\n"
    f"log.csv:\n"
    f" - distinct cmdb_id: {log_cmdbs}\n"
    f" - distinct log_name: {log_names}\n"
    f" - top pairs by record count (examples, up to 5):\n    - " + "\n    - ".join(log_top_examples) + "\n\n"
    f"error_logs.csv:\n"
    f" - total records: {error_total}\n"
    f" - top cmdb_id counts (up to 5):\n    - " + ("\n    - ".join(error_top_examples) if error_top_examples else "none (no error records)") + "\n"
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  count
0         IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31
1587  apache01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31
6         IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31
1583  apache01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31
102       IG02                        OSLinux-CPU_CPU_CPUUserTime     31
3         IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31
1660  apache02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31
1661  apache02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31
101       IG02                         OSLinux-CPU_CPU_CPUSysTime     31
1663  apache02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31
2         IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31
1506  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31
1608  apache02                        OSLinux-CPU_CPU_CPUidleutil     31
1505  Tomcat04     Tomcat-Sessions_7441--_SESSIONRejectedSessions     31
1604  apache02                            OSLinux-CPU_CPU_CPULoad     31
1507  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31
1508  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31
1509  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31
1510  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31
1511  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31,    distinct_cmdb_id  distinct_kpi_name
0                29                351,       cmdb_id                     trace_name  count
263  dockerB2        trace.to_MG02.row_count     30
262  dockerB2     trace.to_MG02.duration_p95     30
261  dockerB2    trace.to_MG02.duration_mean     30
260  dockerB2        trace.to_MG01.row_count     30
3        IG01       trace.self.duration_mean     30
221  dockerA2           trace.self.row_count     30
222  dockerA2    trace.to_MG01.duration_mean     30
223  dockerA2     trace.to_MG01.duration_p95     30
224  dockerA2        trace.to_MG01.row_count     30
225  dockerA2    trace.to_MG02.duration_mean     30
226  dockerA2     trace.to_MG02.duration_p95     30
227  dockerA2        trace.to_MG02.row_count     30
228  dockerB1  trace.from_MG01.duration_mean     30
213  dockerA2  trace.from_MG02.duration_mean     30
214  dockerA2   trace.from_MG02.duration_p95     30
215  dockerA2      trace.from_MG02.row_count     30
219  dockerA2       trace.self.duration_mean     30
220  dockerA2        trace.self.duration_p95     30
205  dockerA1     trace.to_MG01.duration_p95     30
206  dockerA1        trace.to_MG01.row_count     30,    distinct_cmdb_id  distinct_trace_name
0                12                   72,      cmdb_id          log_name  count
4   Tomcat01     log.row_count     31
2   Tomcat01   log.error_count     31
17  apache02     log.row_count     31
15  apache01     log.row_count     31
8   Tomcat03   log.error_count     31
16  apache02   log.error_count     31
14  apache01   log.error_count     31
13  Tomcat04     log.row_count     31
11  Tomcat04   log.error_count     31
10  Tomcat03     log.row_count     31
1    Mysql01     log.row_count     29
0    Mysql01   log.error_count     29
5   Tomcat02   log.error_count     23
7   Tomcat02     log.row_count     23
6   Tomcat02  log.gc_pause_max     23
9   Tomcat03  log.gc_pause_max     15
3   Tomcat01  log.gc_pause_max     14
12  Tomcat04  log.gc_pause_max     14,    distinct_cmdb_id  distinct_log_name
0                 7                  3, 0, Empty DataFrame
Columns: [cmdb_id, count]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing dataframes if present; otherwise load
tz = pytz.timezone('Asia/Shanghai')

try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
    if 'timestamp' in metric_df.columns:
        metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s').dt.tz_localize(tz)

try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')
    if 'timestamp' in trace_df.columns:
        trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s').dt.tz_localize(tz)

try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')
    if 'timestamp' in log_df.columns:
        log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s').dt.tz_localize(tz)

# Candidate components
candidates = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
              "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]

# Helper to compute group P95 and count for a dataframe
def compute_group_p95(df, group_cols, value_col='value'):
    # Ensure value numeric
    df = df.copy()
    df[value_col] = pd.to_numeric(df[value_col], errors='coerce')
    # Filter to candidate components
    df = df[df['cmdb_id'].isin(candidates)]
    if df.shape[0] == 0:
        # Return empty DataFrame with expected columns
        cols = group_cols + ['count', 'p95_value']
        return pd.DataFrame(columns=cols)
    # Group and aggregate
    agg = (
        df.groupby(group_cols, dropna=False)[value_col]
        .agg(count='count', p95_value=lambda s: np.nanpercentile(s.dropna(), 95) if s.dropna().size>0 else np.nan)
        .reset_index()
    )
    # Round p95 for compactness
    agg['p95_value'] = agg['p95_value'].round(6)
    # Sort by p95 descending
    agg = agg.sort_values('p95_value', ascending=False)
    return agg

# 1) metric.csv
metric_thresholds_df = compute_group_p95(metric_df, ['cmdb_id', 'kpi_name'], 'value')
metric_top20_thresholds = metric_thresholds_df.head(20)

# 2) trace.csv
trace_thresholds_df = compute_group_p95(trace_df, ['cmdb_id', 'trace_name'], 'value')
trace_top20_thresholds = trace_thresholds_df.head(20)

# 3) log.csv
log_thresholds_df = compute_group_p95(log_df, ['cmdb_id', 'log_name'], 'value')
log_top20_thresholds = log_thresholds_df.head(20)

# Display the compact top-20 tables (these variables are also stored for later use)
metric_top20_thresholds, trace_top20_thresholds, log_top20_thresholds

```
Out[2]:
```
import pandas as pd

# Reuse top-20 tables computed earlier (should exist in kernel)
# Fallback to empty DataFrames if not present
metric_top20_thresholds = globals().get('metric_top20_thresholds', pd.DataFrame())
trace_top20_thresholds = globals().get('trace_top20_thresholds', pd.DataFrame())
log_top20_thresholds = globals().get('log_top20_thresholds', pd.DataFrame())

def fmt_num(x):
    if pd.isna(x):
        return "NaN"
    # Large integers: show with commas and no decimals
    if abs(x) >= 1000 and float(x).is_integer():
        return f"{int(x):,}"
    # Otherwise show up to 4 significant decimals, trim zeros
    return f"{x:.4f}".rstrip('0').rstrip('.')

def top_examples(df, key_cols, n=5):
    examples = []
    for _, r in df.head(n).iterrows():
        keys = " / ".join(str(r[c]) for c in key_cols)
        examples.append(f"{keys} (count={int(r['count'])}, p95={fmt_num(r['p95_value'])})")
    return examples if examples else ["<none>"]

# Build summary text
summary_lines = []
summary_lines.append("Global P95 thresholds (components limited to the provided candidate list).")
summary_lines.append("")
# Metric
summary_lines.append("1) metric.csv (top pairs by P95):")
if not metric_top20_thresholds.empty:
    summary_lines.append(f" - Showing top {min(20, len(metric_top20_thresholds))} pairs. Examples:")
    for ex in top_examples(metric_top20_thresholds, ['cmdb_id','kpi_name'], 5):
        summary_lines.append(f"    - {ex}")
else:
    summary_lines.append(" - No metric rows for candidate components.")
summary_lines.append("")

# Trace
summary_lines.append("2) trace.csv (top pairs by P95):")
if not trace_top20_thresholds.empty:
    summary_lines.append(f" - Showing top {min(20, len(trace_top20_thresholds))} pairs. Examples:")
    for ex in top_examples(trace_top20_thresholds, ['cmdb_id','trace_name'], 5):
        summary_lines.append(f"    - {ex}")
else:
    summary_lines.append(" - No trace rows for candidate components.")
summary_lines.append("")

# Log
summary_lines.append("3) log.csv (top pairs by P95):")
if not log_top20_thresholds.empty:
    summary_lines.append(f" - Showing top {min(20, len(log_top20_thresholds))} pairs. Examples:")
    for ex in top_examples(log_top20_thresholds, ['cmdb_id','log_name'], 5):
        summary_lines.append(f"    - {ex}")
else:
    summary_lines.append(" - No log rows for candidate components.")
summary_lines.append("")

summary_lines.append("Note: Full threshold tables are stored in kernel variables:")
summary_lines.append(" - metric_thresholds_df  (grouped cmdb_id, kpi_name with count and p95_value)")
summary_lines.append(" - trace_thresholds_df   (grouped cmdb_id, trace_name with count and p95_value)")
summary_lines.append(" - log_thresholds_df     (grouped cmdb_id, log_name with count and p95_value)")

summary = "\n".join(summary_lines)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  count     p95_value
879    Redis01        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn     15  5.682334e+09
880    Redis01       OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut     15  5.155792e+09
914    Redis01  redis-Redis_6379_Redis  (total_commands_proces...     30  5.020394e+09
1018   Redis02                redis-Redis_6379_Redis  (maxmemory)     30  4.294967e+09
907    Redis01                redis-Redis_6379_Redis  (maxmemory)     30  4.294967e+09
1122  Tomcat01             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory      6  4.187619e+09
1124  Tomcat01           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory      6  4.187619e+09
1341  Tomcat03           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory      6  4.187619e+09
1339  Tomcat03             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory      6  4.187619e+09
1448  Tomcat04             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory      6  4.187619e+09
1228  Tomcat02             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory      6  4.187619e+09
1230  Tomcat02           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory      6  4.187619e+09
1450  Tomcat04           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory      6  4.187619e+09
1025   Redis02  redis-Redis_6379_Redis  (total_commands_proces...     30  3.991573e+09
902    Redis01            redis-Redis_6379_Redis  (keyspace_hits)     30  3.815203e+09
1447  Tomcat04            Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory      4  3.216613e+09
1338  Tomcat03            Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory      4  3.039966e+09
1121  Tomcat01            Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory      5  2.888049e+09
1013   Redis02            redis-Redis_6379_Redis  (keyspace_hits)     30  2.885475e+09
1227  Tomcat02            Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory      4  2.864437e+09,       cmdb_id                     trace_name  count  p95_value
131  Tomcat01           trace.self.row_count     30    2332.55
149  Tomcat02           trace.self.row_count     30    2246.15
185  Tomcat04           trace.self.row_count     30    2228.25
167  Tomcat03           trace.self.row_count     30    2178.55
5        IG01           trace.self.row_count     30    1332.10
23       IG02           trace.self.row_count     30    1264.90
107      MG02           trace.self.row_count     30    1020.00
65       MG01           trace.self.row_count     30     975.95
98       MG02  trace.from_dockerB1.row_count     30     207.20
101      MG02  trace.from_dockerB2.row_count     30     198.10
56       MG01  trace.from_dockerB1.row_count     30     198.05
80       MG02  trace.from_Tomcat01.row_count     30     178.85
137  Tomcat01        trace.to_MG02.row_count     30     178.85
125  Tomcat01      trace.from_IG02.row_count     30     175.75
26       IG02    trace.to_Tomcat01.row_count     30     175.75
59       MG01  trace.from_dockerB2.row_count     30     175.45
122  Tomcat01      trace.from_IG01.row_count     30     174.85
8        IG01    trace.to_Tomcat01.row_count     30     174.85
89       MG02  trace.from_Tomcat04.row_count     30     174.00
191  Tomcat04        trace.to_MG02.row_count     30     174.00,      cmdb_id          log_name  count  p95_value
15  apache01     log.row_count     31   654.5000
17  apache02     log.row_count     31   646.0000
4   Tomcat01     log.row_count     31   342.0000
13  Tomcat04     log.row_count     31   334.0000
10  Tomcat03     log.row_count     31   318.0000
1    Mysql01     log.row_count     29     8.2000
7   Tomcat02     log.row_count     23     1.0000
6   Tomcat02  log.gc_pause_max     23     0.0990
3   Tomcat01  log.gc_pause_max     14     0.0835
9   Tomcat03  log.gc_pause_max     15     0.0800
12  Tomcat04  log.gc_pause_max     14     0.0735
0    Mysql01   log.error_count     29     0.0000
2   Tomcat01   log.error_count     31     0.0000
5   Tomcat02   log.error_count     23     0.0000
8   Tomcat03   log.error_count     31     0.0000
11  Tomcat04   log.error_count     31     0.0000
14  apache01   log.error_count     31     0.0000
16  apache02   log.error_count     31     0.0000)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing dataframes/thresholds if present, otherwise load/compute
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Ensure metric_thresholds_df exists (computed globally previously). If missing, compute fallback.
try:
    metric_thresholds_df
except NameError:
    # compute global P95 thresholds for candidate components (safe fallback)
    candidates = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
                  "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
    metric_thresholds_df = (
        metric_df[metric_df['cmdb_id'].isin(candidates)]
        .groupby(['cmdb_id','kpi_name'], dropna=False)['value']
        .agg(count='count', p95_value=lambda s: np.nanpercentile(s.dropna(),95) if s.dropna().size>0 else np.nan)
        .reset_index()
    )
    metric_thresholds_df['p95_value'] = metric_thresholds_df['p95_value'].round(6)

# Candidate components (same list)
candidates = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
              "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]

# Normalize metric_df timestamp to timezone-aware Asia/Shanghai
if 'timestamp' in metric_df.columns:
    if pd.api.types.is_integer_dtype(metric_df['timestamp']) or pd.api.types.is_float_dtype(metric_df['timestamp']):
        metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s').dt.tz_localize(tz)
    elif pd.api.types.is_datetime64_any_dtype(metric_df['timestamp']):
        # If tz-naive, localize; if tz-aware, convert to Asia/Shanghai
        if metric_df['timestamp'].dt.tz is None:
            metric_df['timestamp'] = metric_df['timestamp'].dt.tz_localize(tz)
        else:
            metric_df['timestamp'] = metric_df['timestamp'].dt.tz_convert(tz)
    else:
        # try to parse
        metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', errors='coerce').dt.tz_localize(tz)

# Ensure numeric values
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Incident window: March 9, 2021 18:00 to 18:30 (UTC+8)
start_ts = tz.localize(pd.Timestamp('2021-03-09 18:00:00'))
end_ts   = tz.localize(pd.Timestamp('2021-03-09 18:30:00'))

# Filter metric rows for candidate components and time window (inclusive)
metric_window = metric_df[
    metric_df['cmdb_id'].isin(candidates) &
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts)
].copy()

# Left-join with global thresholds (metric_thresholds_df) on (cmdb_id, kpi_name)
# Ensure metric_thresholds_df has p95_value column
if 'p95_value' not in metric_thresholds_df.columns:
    raise RuntimeError("metric_thresholds_df missing p95_value column")

metric_window = metric_window.merge(
    metric_thresholds_df[['cmdb_id','kpi_name','p95_value']],
    on=['cmdb_id','kpi_name'],
    how='left'
)

# Mark anomalies: value >= p95_value and p95_value not null
metric_window['is_anomaly'] = np.where(
    (~metric_window['p95_value'].isna()) & (~metric_window['value'].isna()) & (metric_window['value'] >= metric_window['p95_value']),
    True, False
)

# Keep only anomaly rows
metric_anomaly_rows = metric_window[metric_window['is_anomaly']].copy()

# Aggregate anomalies by (cmdb_id, kpi_name)
if metric_anomaly_rows.shape[0] == 0:
    # Empty aggregated table with expected columns
    metric_anomalies_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp',
        'max_value_in_window','p95_value','severity_ratio'
    ])
else:
    agg = metric_anomaly_rows.groupby(['cmdb_id','kpi_name'], dropna=False).agg(
        anomaly_count = ('is_anomaly','size'),
        earliest_anomaly_timestamp = ('timestamp','min'),
        max_value_in_window = ('value','max'),
        p95_value = ('p95_value','first')
    ).reset_index()

    # Compute severity_ratio
    def compute_severity(row):
        p95 = row['p95_value']
        mx = row['max_value_in_window']
        if pd.isna(p95):
            return np.nan
        if p95 == 0:
            return np.inf
        return (mx - p95) / p95

    agg['severity_ratio'] = agg.apply(compute_severity, axis=1)

    # Format earliest_anomaly_timestamp as timezone-aware Timestamps (keep as Timestamp type)
    # Round numeric columns for compactness
    agg['max_value_in_window'] = agg['max_value_in_window'].round(6)
    agg['p95_value'] = agg['p95_value'].round(6)
    agg['severity_ratio'] = agg['severity_ratio'].replace([np.inf, -np.inf], np.inf)

    metric_anomalies_df = agg

# Sort by anomaly_count desc, then severity_ratio desc
metric_anomalies_df = metric_anomalies_df.sort_values(['anomaly_count','severity_ratio'], ascending=[False, False]).reset_index(drop=True)

# Prepare compact top-20 table to display with selected columns
display_cols = ['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','p95_value','severity_ratio']
metric_anomalies_top20 = metric_anomalies_df[display_cols].head(20)

# Ensure earliest_anomaly_timestamp displayed in UTC+8 timezone (it should already be)
# Final output variable (kernel will display it)
metric_anomalies_top20

```
Out[3]:
```
# Summarize the anomaly detection result as a plain-English string (compact)
try:
    n_checked = int(metric_window.shape[0])
except Exception:
    n_checked = "unknown"

try:
    n_anomaly_groups = int(metric_anomalies_df.shape[0])
except Exception:
    n_anomaly_groups = 0

summary = (
    "Summary of metric anomaly detection for 2021-03-09 18:00–18:30 (UTC+8):\n"
    f" - Metric rows evaluated (candidate components): {n_checked}\n"
    f" - Aggregated anomaly groups found: {n_anomaly_groups}\n"
    " - Conclusion: No metric anomaly groups were detected in the incident window "
    "when comparing values against the precomputed global P95 thresholds.\n"
    " - Notes: The full aggregated anomaly table is stored in kernel variable "
    "'metric_anomalies_df' (empty in this run)."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [cmdb_id, kpi_name, anomaly_count, earliest_anomaly_timestamp, max_value_in_window, p95_value, severity_ratio]
Index: []```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing trace_df and trace_thresholds_df if available; otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

try:
    trace_thresholds_df
except NameError:
    # Fallback: compute global P95 thresholds for candidate components from full trace.csv
    candidates = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
                  "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]
    trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
    trace_thresholds_df = (
        trace_df[trace_df['cmdb_id'].isin(candidates)]
        .groupby(['cmdb_id','trace_name'], dropna=False)['value']
        .agg(count='count', p95_value=lambda s: np.nanpercentile(s.dropna(),95) if s.dropna().size>0 else np.nan)
        .reset_index()
    )
    trace_thresholds_df['p95_value'] = trace_thresholds_df['p95_value'].round(6)

# Candidate components
candidates = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
              "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]

# Normalize timestamp column to timezone-aware Asia/Shanghai
if 'timestamp' in trace_df.columns:
    if pd.api.types.is_integer_dtype(trace_df['timestamp']) or pd.api.types.is_float_dtype(trace_df['timestamp']):
        trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s').dt.tz_localize(tz)
    elif pd.api.types.is_datetime64_any_dtype(trace_df['timestamp']):
        if trace_df['timestamp'].dt.tz is None:
            trace_df['timestamp'] = trace_df['timestamp'].dt.tz_localize(tz)
        else:
            trace_df['timestamp'] = trace_df['timestamp'].dt.tz_convert(tz)
    else:
        trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', errors='coerce').dt.tz_localize(tz)

# Ensure numeric values
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Incident window (inclusive) in UTC+8
start_ts = pd.Timestamp('2021-03-09 18:00:00')
end_ts   = pd.Timestamp('2021-03-09 18:30:00')
start_ts = tz.localize(start_ts)
end_ts = tz.localize(end_ts)

# Filter to candidate components and incident window
trace_window = trace_df[
    trace_df['cmdb_id'].isin(candidates) &
    (trace_df['timestamp'] >= start_ts) &
    (trace_df['timestamp'] <= end_ts)
].copy()

# Left join with global thresholds
if 'p95_value' not in trace_thresholds_df.columns:
    raise RuntimeError("trace_thresholds_df missing p95_value column")

trace_window = trace_window.merge(
    trace_thresholds_df[['cmdb_id','trace_name','p95_value']],
    on=['cmdb_id','trace_name'],
    how='left'
)

# Ignore rows where p95_value is missing
trace_window_valid = trace_window[~trace_window['p95_value'].isna()].copy()

# Mark anomalies: value >= p95_value
trace_window_valid['is_anomaly'] = np.where(
    (~trace_window_valid['value'].isna()) & (trace_window_valid['value'] >= trace_window_valid['p95_value']),
    True, False
)

trace_anomaly_rows = trace_window_valid[trace_window_valid['is_anomaly']].copy()

# Aggregate anomalies by (cmdb_id, trace_name)
if trace_anomaly_rows.shape[0] == 0:
    trace_anomalies_df = pd.DataFrame(columns=[
        'cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp',
        'max_value_in_window','p95_value','severity_ratio'
    ])
else:
    agg = trace_anomaly_rows.groupby(['cmdb_id','trace_name'], dropna=False).agg(
        anomaly_count = ('is_anomaly','size'),
        earliest_anomaly_timestamp = ('timestamp','min'),
        max_value_in_window = ('value','max'),
        p95_value = ('p95_value','first')
    ).reset_index()

    def compute_severity(row):
        p95 = row['p95_value']
        mx = row['max_value_in_window']
        if pd.isna(p95):
            return np.nan
        if p95 == 0:
            return np.inf
        return (mx - p95) / p95

    agg['severity_ratio'] = agg.apply(compute_severity, axis=1)
    agg['max_value_in_window'] = agg['max_value_in_window'].round(6)
    agg['p95_value'] = agg['p95_value'].round(6)
    agg['severity_ratio'] = agg['severity_ratio'].replace([np.inf, -np.inf], np.inf)

    trace_anomalies_df = agg

# Sort and prepare compact top-20
trace_anomalies_df = trace_anomalies_df.sort_values(['anomaly_count','severity_ratio'], ascending=[False, False]).reset_index(drop=True)
trace_anomalies_top20 = trace_anomalies_df[['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','p95_value','severity_ratio']].head(20)

# Display compact top-20 table (stored full results in trace_anomalies_df)
trace_anomalies_top20

```
Out[4]:
```
summary = (
    "Trace anomaly detection for 2021-03-09 18:00–18:30 (UTC+8):\n"
    f" - Trace rows evaluated (candidate components): {trace_window.shape[0] if 'trace_window' in globals() else 'unknown'}\n"
    f" - Aggregated anomaly groups found: {trace_anomalies_df.shape[0] if 'trace_anomalies_df' in globals() else 'unknown'}\n"
    " - Conclusion: No trace anomaly groups were detected in the incident window when comparing values against the precomputed global P95 thresholds.\n"
    " - Note: The full aggregated anomaly table is stored in kernel variable 'trace_anomalies_df' (empty in this run)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [cmdb_id, trace_name, anomaly_count, earliest_anomaly_timestamp, max_value_in_window, p95_value, severity_ratio]
Index: []```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing log_df and log_thresholds_df if present
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')

try:
    log_thresholds_df
except NameError:
    # Fallback: compute global P95 thresholds for candidate components from full log.csv
    candidates = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
                  "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]
    log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')
    log_thresholds_df = (
        log_df[log_df['cmdb_id'].isin(candidates)]
        .groupby(['cmdb_id','log_name'], dropna=False)['value']
        .agg(count='count', p95_value=lambda s: np.nanpercentile(s.dropna(),95) if s.dropna().size>0 else np.nan)
        .reset_index()
    )
    log_thresholds_df['p95_value'] = log_thresholds_df['p95_value'].round(6)

# Candidate components
candidates = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
              "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]

# Normalize timestamp in log_df to Asia/Shanghai tz-aware datetimes
if 'timestamp' in log_df.columns:
    if pd.api.types.is_integer_dtype(log_df['timestamp']) or pd.api.types.is_float_dtype(log_df['timestamp']):
        log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s').dt.tz_localize(tz)
    elif pd.api.types.is_datetime64_any_dtype(log_df['timestamp']):
        if log_df['timestamp'].dt.tz is None:
            log_df['timestamp'] = log_df['timestamp'].dt.tz_localize(tz)
        else:
            log_df['timestamp'] = log_df['timestamp'].dt.tz_convert(tz)
    else:
        log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', errors='coerce').dt.tz_localize(tz)

# Ensure numeric values
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# Incident window (inclusive) in UTC+8
start_ts = tz.localize(pd.Timestamp('2021-03-09 18:00:00'))
end_ts   = tz.localize(pd.Timestamp('2021-03-09 18:30:00'))

# Filter to candidate components and incident window
log_window = log_df[
    log_df['cmdb_id'].isin(candidates) &
    (log_df['timestamp'] >= start_ts) &
    (log_df['timestamp'] <= end_ts)
].copy()

# Left join with precomputed global thresholds
if 'p95_value' not in log_thresholds_df.columns:
    raise RuntimeError("log_thresholds_df missing p95_value column")

log_window = log_window.merge(
    log_thresholds_df[['cmdb_id','log_name','p95_value']],
    on=['cmdb_id','log_name'],
    how='left'
)

# Ignore rows with missing p95_value
log_window_valid = log_window[~log_window['p95_value'].isna()].copy()

# Mark anomalies: value >= p95_value
log_window_valid['is_anomaly'] = np.where(
    (~log_window_valid['value'].isna()) & (log_window_valid['value'] >= log_window_valid['p95_value']),
    True, False
)

log_anomaly_rows = log_window_valid[log_window_valid['is_anomaly']].copy()

# Aggregate anomalies by (cmdb_id, log_name)
if log_anomaly_rows.shape[0] == 0:
    log_anomalies_df = pd.DataFrame(columns=[
        'cmdb_id','log_name','anomaly_count','earliest_anomaly_timestamp',
        'max_value_in_window','p95_value','severity_ratio'
    ])
else:
    agg = log_anomaly_rows.groupby(['cmdb_id','log_name'], dropna=False).agg(
        anomaly_count = ('is_anomaly','size'),
        earliest_anomaly_timestamp = ('timestamp','min'),
        max_value_in_window = ('value','max'),
        p95_value = ('p95_value','first')
    ).reset_index()

    def compute_severity(row):
        p95 = row['p95_value']
        mx = row['max_value_in_window']
        if pd.isna(p95):
            return np.nan
        if p95 == 0:
            return np.inf
        return (mx - p95) / p95

    agg['severity_ratio'] = agg.apply(compute_severity, axis=1)
    agg['max_value_in_window'] = agg['max_value_in_window'].round(6)
    agg['p95_value'] = agg['p95_value'].round(6)
    agg['severity_ratio'] = agg['severity_ratio'].replace([np.inf, -np.inf], np.inf)

    log_anomalies_df = agg

# Sort by anomaly_count desc then severity_ratio desc and prepare compact top-20
log_anomalies_df = log_anomalies_df.sort_values(['anomaly_count','severity_ratio'], ascending=[False, False]).reset_index(drop=True)
log_anomalies_top20 = log_anomalies_df[['cmdb_id','log_name','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','p95_value','severity_ratio']].head(20)

# Display the compact top-20 table (full aggregated result stored in log_anomalies_df)
log_anomalies_top20

```
Out[5]:
```
summary = (
    "Log anomaly detection for 2021-03-09 18:00–18:30 (UTC+8):\n"
    f" - Log rows evaluated (candidate components): {log_window.shape[0] if 'log_window' in globals() else 'unknown'}\n"
    f" - Aggregated anomaly groups found: {log_anomalies_df.shape[0] if 'log_anomalies_df' in globals() else 'unknown'}\n"
    " - Conclusion: No log anomaly groups were detected in the incident window when comparing values against the precomputed global P95 thresholds.\n"
    " - Note: The full aggregated anomaly table is stored in kernel variable 'log_anomalies_df' (empty in this run)."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [cmdb_id, log_name, anomaly_count, earliest_anomaly_timestamp, max_value_in_window, p95_value, severity_ratio]
Index: []```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing trace_df if available; otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

# Candidate components
candidates = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
              "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]

# Ensure timestamp is timezone-aware in Asia/Shanghai (UTC+8)
if 'timestamp' in trace_df.columns:
    if pd.api.types.is_integer_dtype(trace_df['timestamp']) or pd.api.types.is_float_dtype(trace_df['timestamp']):
        trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s').dt.tz_localize(tz)
    elif pd.api.types.is_datetime64_any_dtype(trace_df['timestamp']):
        if trace_df['timestamp'].dt.tz is None:
            trace_df['timestamp'] = trace_df['timestamp'].dt.tz_localize(tz)
        else:
            trace_df['timestamp'] = trace_df['timestamp'].dt.tz_convert(tz)
    else:
        trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', errors='coerce').dt.tz_localize(tz)

# Ensure numeric 'value'
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# 1) Compute global P15 and P5 per (cmdb_id, trace_name) using the full trace_df (before filtering)
trace_full_candidates = trace_df[trace_df['cmdb_id'].isin(candidates)].copy()

def p15(s):
    s2 = s.dropna()
    return np.nanpercentile(s2, 15) if s2.size > 0 else np.nan

def p5(s):
    s2 = s.dropna()
    return np.nanpercentile(s2, 5) if s2.size > 0 else np.nan

trace_low_thresholds_df = (
    trace_full_candidates
    .groupby(['cmdb_id','trace_name'], dropna=False)['value']
    .agg(count='count',
         p15_value=lambda s: p15(s),
         p5_value=lambda s: p5(s))
    .reset_index()
)

# Round thresholds for compactness
trace_low_thresholds_df['p15_value'] = trace_low_thresholds_df['p15_value'].round(6)
trace_low_thresholds_df['p5_value'] = trace_low_thresholds_df['p5_value'].round(6)

# 2) Filter trace.csv to incident window and candidate components
start_ts = tz.localize(pd.Timestamp('2021-03-09 18:00:00'))
end_ts   = tz.localize(pd.Timestamp('2021-03-09 18:30:00'))

trace_window = trace_df[
    trace_df['cmdb_id'].isin(candidates) &
    (trace_df['timestamp'] >= start_ts) &
    (trace_df['timestamp'] <= end_ts)
].copy()

# 3) Left-join each row with global p15 and p5 thresholds
trace_window = trace_window.merge(
    trace_low_thresholds_df[['cmdb_id','trace_name','p15_value','p5_value']],
    on=['cmdb_id','trace_name'],
    how='left'
)

# 4) Ignore rows with missing thresholds
trace_window_valid = trace_window[~trace_window['p15_value'].isna() & ~trace_window['p5_value'].isna()].copy()

# 5) Mark low-tail anomalies: value <= p15_value, and flag p5
trace_window_valid['is_low_anomaly'] = np.where(
    (~trace_window_valid['value'].isna()) & (trace_window_valid['value'] <= trace_window_valid['p15_value']),
    True, False
)
trace_window_valid['is_p5_or_below'] = np.where(
    (~trace_window_valid['value'].isna()) & (trace_window_valid['value'] <= trace_window_valid['p5_value']),
    True, False
)

# Keep only low-tail anomaly rows
trace_low_rows = trace_window_valid[trace_window_valid['is_low_anomaly']].copy()

# 6) Aggregate low-tail anomalies by (cmdb_id, trace_name)
if trace_low_rows.shape[0] == 0:
    trace_low_anomalies_df = pd.DataFrame(columns=[
        'cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp',
        'min_value_in_window','p15_value','p5_value','severity_ratio'
    ])
else:
    agg = trace_low_rows.groupby(['cmdb_id','trace_name'], dropna=False).agg(
        anomaly_count = ('is_low_anomaly','size'),
        earliest_anomaly_timestamp = ('timestamp','min'),
        min_value_in_window = ('value','min'),
        p15_value = ('p15_value','first'),
        p5_value = ('p5_value','first')
    ).reset_index()

    def compute_severity(row):
        p15v = row['p15_value']
        mn = row['min_value_in_window']
        if pd.isna(p15v):
            return np.nan
        if p15v == 0:
            return np.inf
        return (p15v - mn) / p15v

    agg['severity_ratio'] = agg.apply(compute_severity, axis=1)

    # Rounding for compactness
    agg['min_value_in_window'] = agg['min_value_in_window'].round(6)
    agg['p15_value'] = agg['p15_value'].round(6)
    agg['p5_value'] = agg['p5_value'].round(6)
    agg['severity_ratio'] = agg['severity_ratio'].replace([np.inf, -np.inf], np.inf)

    trace_low_anomalies_df = agg

# 7) Store full aggregated table in kernel variable trace_low_anomalies_df (already set)
# 8) Prepare compact top-20 table ordered by anomaly_count desc then severity_ratio desc
trace_low_anomalies_df = trace_low_anomalies_df.sort_values(['anomaly_count','severity_ratio'], ascending=[False, False]).reset_index(drop=True)
trace_low_anomalies_top20 = trace_low_anomalies_df[
    ['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp','min_value_in_window','p15_value','p5_value','severity_ratio']
].head(20)

# Display compact top-20 aggregated results
trace_low_anomalies_top20

```
Out[6]:
```
summary = (
    "Low-tail (drop) anomaly detection for trace series on 2021-03-09 18:00–18:30 (UTC+8):\n"
    f" - Candidate components checked: 14 (apache01, apache02, Tomcat01-04, MG01-02, IG01-02, Mysql01-02, Redis01-02)\n"
    f" - Trace rows in the incident window (candidate components): {trace_window.shape[0] if 'trace_window' in globals() else 'unknown'}\n"
    f" - Aggregated low-tail anomaly groups found: {trace_low_anomalies_df.shape[0] if 'trace_low_anomalies_df' in globals() else 'unknown'}\n"
    " - Conclusion: No low-tail trace anomaly groups were detected in the incident window when comparing values against the global P15/P5 thresholds.\n"
    " - Notes: Global P15/P5 thresholds (per cmdb_id, trace_name) are stored in 'trace_low_thresholds_df'.\n"
    "   The full aggregated low-tail anomaly table (empty in this run) is stored in 'trace_low_anomalies_df'."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [cmdb_id, trace_name, anomaly_count, earliest_anomaly_timestamp, min_value_in_window, p15_value, p5_value, severity_ratio]
Index: []```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing dataframes if present, otherwise load from CSVs
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')
try:
    errors_df
except NameError:
    errors_df = pd.read_csv('error_logs.csv')

# Standardize timestamp columns to timezone-aware Asia/Shanghai (UTC+8)
def ensure_tz(df, ts_col='timestamp'):
    if ts_col not in df.columns:
        return df
    col = df[ts_col]
    if pd.api.types.is_integer_dtype(col) or pd.api.types.is_float_dtype(col):
        df[ts_col] = pd.to_datetime(col, unit='s', errors='coerce').dt.tz_localize(tz)
    elif pd.api.types.is_datetime64_any_dtype(col):
        if col.dt.tz is None:
            df[ts_col] = col.dt.tz_localize(tz)
        else:
            df[ts_col] = col.dt.tz_convert(tz)
    else:
        # try to parse as epoch seconds fallback
        df[ts_col] = pd.to_datetime(col, unit='s', errors='coerce').dt.tz_localize(tz)
    return df

metric_df = ensure_tz(metric_df, 'timestamp')
trace_df  = ensure_tz(trace_df, 'timestamp')
log_df    = ensure_tz(log_df, 'timestamp')
errors_df = ensure_tz(errors_df, 'timestamp')

# Candidate components list
candidates = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
              "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]

# Incident window (inclusive) in UTC+8
start_ts = tz.localize(pd.Timestamp('2021-03-09 18:00:00'))
end_ts   = tz.localize(pd.Timestamp('2021-03-09 18:30:00'))

# Filter function
def filter_window(df, ts_col='timestamp', comp_col='cmdb_id'):
    if ts_col not in df.columns or comp_col not in df.columns:
        return df.iloc[0:0].copy()  # empty df with same columns
    return df[
        df[comp_col].isin(candidates) &
        (df[ts_col] >= start_ts) &
        (df[ts_col] <= end_ts)
    ].copy()

# Create and store filtered windows in kernel variables
metric_window = filter_window(metric_df, 'timestamp', 'cmdb_id')
trace_window  = filter_window(trace_df,  'timestamp', 'cmdb_id')
log_window    = filter_window(log_df,    'timestamp', 'cmdb_id')
error_window  = filter_window(errors_df, 'timestamp', 'cmdb_id')

# Aggregated summaries: total counts and top-20 counts per cmdb_id
def top_counts(df, comp_col='cmdb_id', topn=20):
    if comp_col not in df.columns:
        return pd.DataFrame(columns=[comp_col, 'count'])
    counts = df.groupby(comp_col).size().reset_index(name='count').sort_values('count', ascending=False).head(topn)
    return counts

metric_total = metric_window.shape[0]
metric_counts_top20 = top_counts(metric_window)

trace_total = trace_window.shape[0]
trace_counts_top20 = top_counts(trace_window)

log_total = log_window.shape[0]
log_counts_top20 = top_counts(log_window)

error_total = error_window.shape[0]
error_counts_top20 = top_counts(error_window)

# For error_logs: include up to 5 example messages per top cmdb_id (truncate messages)
def example_messages(df, top_counts_df, msg_col='message', examples_per=5, truncate_chars=200):
    rows = []
    for _, r in top_counts_df.iterrows():
        cmdb = r['cmdb_id']
        subset = df[df['cmdb_id'] == cmdb]
        if msg_col not in subset.columns:
            examples = []
        else:
            msgs = subset[msg_col].dropna().astype(str).head(examples_per).tolist()
            def trunc(s):
                return (s[:truncate_chars] + '...') if len(s) > truncate_chars else s
            examples = [trunc(m) for m in msgs]
        examples_str = " | ".join(examples) if examples else ""
        rows.append({'cmdb_id': cmdb, 'count': int(r['count']), 'example_messages': examples_str})
    return pd.DataFrame(rows)

error_examples_top20 = example_messages(error_window, error_counts_top20, msg_col='message', examples_per=5, truncate_chars=200)

# Keep final output compact: totals and top-20 counts, plus error examples
metric_total, metric_counts_top20, trace_total, trace_counts_top20, log_total, log_counts_top20, error_total, error_counts_top20, error_examples_top20

```
Out[7]:
```
summary = (
    "Filtered telemetry summary for 2021-03-09 18:00–18:30 (UTC+8) for the specified candidate components:\n"
    "- metric.csv: 0 rows in window (metric_window empty). Top per-cmdb_id counts: none.\n"
    "- trace.csv: 0 rows in window (trace_window empty). Top per-cmdb_id counts: none.\n"
    "- log.csv: 0 rows in window (log_window empty). Top per-cmdb_id counts: none.\n"
    "- error_logs.csv: 0 rows in window (error_window empty). No example messages available.\n\n"
    "Conclusion: No telemetry rows matched the candidate components during the incident window. "
    "If you expected data, verify the files' timestamps are in seconds since epoch and aligned to UTC+8, "
    "and that component names match the candidate list exactly."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(0, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], 0, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], 0, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], 0, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: []
Index: [])```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing dataframes if present; otherwise load from CSVs
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')
try:
    errors_df
except NameError:
    errors_df = pd.read_csv('error_logs.csv')

# Helper to normalize/ensure timestamp column is tz-aware in Asia/Shanghai
def ensure_timestamp_tz(df, ts_col='timestamp'):
    df = df.copy()
    if ts_col not in df.columns:
        return df
    col = df[ts_col]
    # If numeric (epoch seconds), parse with unit='s' then localize
    if pd.api.types.is_integer_dtype(col) or pd.api.types.is_float_dtype(col):
        df[ts_col] = pd.to_datetime(col, unit='s', errors='coerce').dt.tz_localize(tz)
    else:
        # Try to coerce to datetime first
        coerced = pd.to_datetime(col, errors='coerce')
        if coerced.dt.tz is None:
            # tz-naive -> localize to Asia/Shanghai
            df[ts_col] = coerced.dt.tz_localize(tz)
        else:
            # tz-aware -> convert to Asia/Shanghai
            df[ts_col] = coerced.dt.tz_convert(tz)
    return df

# Normalize timestamps for all files
metric_df = ensure_timestamp_tz(metric_df, 'timestamp')
trace_df  = ensure_timestamp_tz(trace_df,  'timestamp')
log_df    = ensure_timestamp_tz(log_df,    'timestamp')
errors_df = ensure_timestamp_tz(errors_df, 'timestamp')

# Candidate components (for reference)
candidates = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
              "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]

# Helper to build per-file summary
def file_summary(df, file_label, ts_col='timestamp', comp_col='cmdb_id', top_n=10, example_n=3, list_limit=50):
    total_rows = int(df.shape[0])
    # min/max timestamps (as ISO strings) or None
    if ts_col in df.columns and not df[ts_col].isna().all():
        min_ts = df[ts_col].min()
        max_ts = df[ts_col].max()
        # Format as ISO strings with timezone info
        min_ts_s = pd.Timestamp(min_ts).isoformat()
        max_ts_s = pd.Timestamp(max_ts).isoformat()
    else:
        min_ts_s = None
        max_ts_s = None
    # distinct cmdb ids
    if comp_col in df.columns:
        distinct_ids = pd.Index(df[comp_col].dropna().unique())
        distinct_count = int(distinct_ids.size)
        distinct_list = distinct_ids[:list_limit].tolist()
    else:
        distinct_count = 0
        distinct_list = []
    # top N cmdb_id by count
    if comp_col in df.columns and total_rows>0:
        top_counts = df.groupby(comp_col).size().reset_index(name='count').sort_values('count', ascending=False).head(top_n)
    else:
        top_counts = pd.DataFrame(columns=[comp_col, 'count'])
    # example rows (first example_n rows) - keep columns compact
    examples = df.head(example_n).copy()
    # For examples, format timestamp column as ISO string if present
    if ts_col in examples.columns:
        examples[ts_col] = examples[ts_col].apply(lambda x: pd.Timestamp(x).isoformat() if pd.notna(x) else None)
    return {
        'file': file_label,
        'total_rows': total_rows,
        'min_timestamp': min_ts_s,
        'max_timestamp': max_ts_s,
        'distinct_cmdb_id_count': distinct_count,
        'distinct_cmdb_ids_sample': distinct_list,
        'top_counts': top_counts,
        'examples': examples
    }

# Build summaries for all four files
metric_summary = file_summary(metric_df, 'metric.csv', ts_col='timestamp', comp_col='cmdb_id', top_n=10, example_n=3)
trace_summary  = file_summary(trace_df,  'trace.csv',  ts_col='timestamp', comp_col='cmdb_id', top_n=10, example_n=3)
log_summary    = file_summary(log_df,    'log.csv',    ts_col='timestamp', comp_col='cmdb_id', top_n=10, example_n=3)
errors_summary = file_summary(errors_df, 'error_logs.csv', ts_col='timestamp', comp_col='cmdb_id', top_n=10, example_n=3)

# Prepare compact DataFrames for display: summary_table and top_counts & examples per file
summary_table = pd.DataFrame([
    {
        'file': metric_summary['file'],
        'total_rows': metric_summary['total_rows'],
        'min_timestamp': metric_summary['min_timestamp'],
        'max_timestamp': metric_summary['max_timestamp'],
        'distinct_cmdb_id_count': metric_summary['distinct_cmdb_id_count'],
        'distinct_cmdb_ids_sample': ", ".join(metric_summary['distinct_cmdb_ids_sample'])
    },
    {
        'file': trace_summary['file'],
        'total_rows': trace_summary['total_rows'],
        'min_timestamp': trace_summary['min_timestamp'],
        'max_timestamp': trace_summary['max_timestamp'],
        'distinct_cmdb_id_count': trace_summary['distinct_cmdb_id_count'],
        'distinct_cmdb_ids_sample': ", ".join(trace_summary['distinct_cmdb_ids_sample'])
    },
    {
        'file': log_summary['file'],
        'total_rows': log_summary['total_rows'],
        'min_timestamp': log_summary['min_timestamp'],
        'max_timestamp': log_summary['max_timestamp'],
        'distinct_cmdb_id_count': log_summary['distinct_cmdb_id_count'],
        'distinct_cmdb_ids_sample': ", ".join(log_summary['distinct_cmdb_ids_sample'])
    },
    {
        'file': errors_summary['file'],
        'total_rows': errors_summary['total_rows'],
        'min_timestamp': errors_summary['min_timestamp'],
        'max_timestamp': errors_summary['max_timestamp'],
        'distinct_cmdb_id_count': errors_summary['distinct_cmdb_id_count'],
        'distinct_cmdb_ids_sample': ", ".join(errors_summary['distinct_cmdb_ids_sample'])
    }
])

# Top counts and examples DataFrames for display (compact)
metric_top10 = metric_summary['top_counts']
metric_examples = metric_summary['examples']

trace_top10 = trace_summary['top_counts']
trace_examples = trace_summary['examples']

log_top10 = log_summary['top_counts']
log_examples = log_summary['examples']

errors_top10 = errors_summary['top_counts']
errors_examples = errors_summary['examples']

# Display compact results (variables returned; Controller will show them)
summary_table, metric_top10, metric_examples, trace_top10, trace_examples, log_top10, log_examples, errors_top10, errors_examples

```
Out[8]:
```
Summary of the four telemetry files (compact):

Overall note
- All files contain data for 2021-03-09 between 10:00 and ~10:30 (Asia/Shanghai / UTC+8). The incident window you requested (2021-03-09 18:00–18:30 UTC+8) does NOT overlap the timestamp ranges in these files, so earlier filtered-window checks returned empty results.

metric.csv
- total_rows: 27,644
- min_timestamp: 2021-03-09T10:00:00+08:00
- max_timestamp: 2021-03-09T10:30:00+08:00
- distinct_cmdb_id_count: 29
- sample distinct cmdb_ids (truncated): IG01, IG02, MG01, MG02, Mysql01, Mysql02, Redis01, Redis02, Tomcat01, Tomcat02, ...
- top 10 cmdb_id by row count (cmdb_id, count):
  - Mysql01: 4008
  - Mysql02: 3906
  - Redis01: 2224
  - Redis02: 2219
  - Tomcat04: 1595
  - Tomcat02: 1563
  - Tomcat03: 1502
  - Tomcat01: 1446
  - IG02: 1439
  - IG01: 1431
- up to 3 example rows (first 3):
  - 2021-03-09T10:00:00+08:00, IG01, JVM-Memory_7778_JVM_Memory_HeapMemoryMax, 2.040136e+09
  - 2021-03-09T10:00:00+08:00, IG01, JVM-Memory_7778_JVM_Memory_HeapMemoryUsage, 8.158200e+00
  - 2021-03-09T10:00:00+08:00, IG01, JVM-Memory_7778_JVM_Memory_HeapMemoryUsed, 3.520943e+08

trace.csv
- total_rows: 6,996
- min_timestamp: 2021-03-09T10:00:00+08:00
- max_timestamp: 2021-03-09T10:29:00+08:00
- distinct_cmdb_id_count: 12
- sample distinct cmdb_ids (truncated): IG01, IG02, MG01, MG02, Tomcat01, Tomcat02, Tomcat03, Tomcat04, dockerA1, dockerA2, dockerB1, dockerB2
- top 10 cmdb_id by row count (cmdb_id, count):
  - MG02: 1185
  - MG01: 1185
  - IG02: 465
  - IG01: 465
  - Tomcat04: 465
  - dockerB1: 465
  - Tomcat02: 462
  - Tomcat01: 462
  - dockerA2: 462
  - dockerA1: 462
- up to 3 example rows (first 3):
  - 2021-03-09T10:00:00+08:00, IG01, trace.from_root.duration_mean, 0.229312
  - 2021-03-09T10:00:00+08:00, IG01, trace.from_root.duration_p95, 1.056250
  - 2021-03-09T10:00:00+08:00, IG01, trace.from_root.row_count, 16.000000

log.csv
- total_rows: 480
- min_timestamp: 2021-03-09T10:00:00+08:00
- max_timestamp: 2021-03-09T10:30:00+08:00
- distinct_cmdb_id_count: 7
- sample distinct cmdb_ids (truncated): Mysql01, Tomcat01, Tomcat02, Tomcat03, Tomcat04, apache01, apache02
- top 10 cmdb_id by row count (cmdb_id, count):
  - Tomcat03: 77
  - Tomcat04: 76
  - Tomcat01: 76
  - Tomcat02: 69
  - apache01: 62
  - apache02: 62
  - Mysql01: 58
- up to 3 example rows (first 3):
  - 2021-03-09T10:00:00+08:00, Mysql01, log.error_count, 0.0
  - 2021-03-09T10:00:00+08:00, Mysql01, log.row_count, 4.0
  - 2021-03-09T10:00:00+08:00, Tomcat01, log.error_count, 0.0

error_logs.csv
- total_rows: 0
- min_timestamp: None
- max_timestamp: None
- distinct_cmdb_id_count: 0
- distinct cmdb_id sample: (none)
- top cmdb_id by count: none
- example rows: none

Key conclusion and next steps
- The telemetry files contain data around 10:00–10:30 (UTC+8), not the 18:00–18:30 incident window you requested. That explains why earlier window-filtered anomaly checks returned empty results.
- Verify whether the incident timestamps are correct (maybe a different day/time) or whether the source CSVs contain the full day of data. Also confirm that the candidate component names match exactly those in the files (they do appear in the distinct lists for metric/trace/log).
- If you want, I can:
  - re-run detection for the 10:00–10:30 window, or
  - scan the CSVs for other timestamp ranges containing 18:00–18:30, or
  - search for rows where cmdb_id exactly matches any candidate and list their timestamp ranges.

The original code execution output of IPython Kernel is also provided below for reference:

(             file  total_rows              min_timestamp              max_timestamp  distinct_cmdb_id_count                           distinct_cmdb_ids_sample
0      metric.csv       27644  2021-03-09T10:00:00+08:00  2021-03-09T10:30:00+08:00                      29  IG01, IG02, MG01, MG02, Mysql01, Mysql02, Redi...
1       trace.csv        6996  2021-03-09T10:00:00+08:00  2021-03-09T10:29:00+08:00                      12  IG01, IG02, MG01, MG02, Tomcat01, Tomcat02, To...
2         log.csv         480  2021-03-09T10:00:00+08:00  2021-03-09T10:30:00+08:00                       7  Mysql01, Tomcat01, Tomcat02, Tomcat03, Tomcat0...
3  error_logs.csv           0                       None                       None                       0                                                   ,      cmdb_id  count
4    Mysql01   4008
5    Mysql02   3906
6    Redis01   2224
7    Redis02   2219
22  Tomcat04   1595
20  Tomcat02   1563
21  Tomcat03   1502
19  Tomcat01   1446
1       IG02   1439
0       IG01   1431,                    timestamp cmdb_id                                    kpi_name         value
0  2021-03-09T10:00:00+08:00    IG01    JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09
1  2021-03-09T10:00:00+08:00    IG01  JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  8.158200e+00
2  2021-03-09T10:00:00+08:00    IG01   JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  3.520943e+08,      cmdb_id  count
3       MG02   1185
2       MG01   1185
1       IG02    465
0       IG01    465
7   Tomcat04    465
10  dockerB1    465
5   Tomcat02    462
4   Tomcat01    462
9   dockerA2    462
8   dockerA1    462,                    timestamp cmdb_id                     trace_name      value
0  2021-03-09T10:00:00+08:00    IG01  trace.from_root.duration_mean   0.229312
1  2021-03-09T10:00:00+08:00    IG01   trace.from_root.duration_p95   1.056250
2  2021-03-09T10:00:00+08:00    IG01      trace.from_root.row_count  16.000000,     cmdb_id  count
3  Tomcat03     77
4  Tomcat04     76
1  Tomcat01     76
2  Tomcat02     69
5  apache01     62
6  apache02     62
0   Mysql01     58,                    timestamp   cmdb_id         log_name  value
0  2021-03-09T10:00:00+08:00   Mysql01  log.error_count    0.0
1  2021-03-09T10:00:00+08:00   Mysql01    log.row_count    4.0
2  2021-03-09T10:00:00+08:00  Tomcat01  log.error_count    0.0, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Candidate components
candidates = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
              "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]

# Reuse loaded dataframes if present, otherwise load from CSV
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')

# Ensure timestamps are tz-aware in Asia/Shanghai (UTC+8)
def ensure_tz(df, ts_col='timestamp'):
    if ts_col not in df.columns:
        return df
    col = df[ts_col]
    # parse numeric epoch seconds
    if pd.api.types.is_integer_dtype(col) or pd.api.types.is_float_dtype(col):
        df[ts_col] = pd.to_datetime(col, unit='s', errors='coerce').dt.tz_localize(tz)
    else:
        coerced = pd.to_datetime(col, errors='coerce')
        if coerced.dt.tz is None:
            df[ts_col] = coerced.dt.tz_localize(tz)
        else:
            df[ts_col] = coerced.dt.tz_convert(tz)
    return df

metric_df = ensure_tz(metric_df, 'timestamp')
trace_df  = ensure_tz(trace_df, 'timestamp')
log_df    = ensure_tz(log_df, 'timestamp')

# Ensure numeric 'value' columns
metric_df['value'] = pd.to_numeric(metric_df.get('value'), errors='coerce')
trace_df['value']  = pd.to_numeric(trace_df.get('value'), errors='coerce')
log_df['value']    = pd.to_numeric(log_df.get('value'), errors='coerce')

# Ensure global thresholds exist (compute if missing) BEFORE filtering window
# Metric p95
try:
    metric_thresholds_df
except NameError:
    metric_full_candidates = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()
    metric_thresholds_df = (
        metric_full_candidates
        .groupby(['cmdb_id','kpi_name'], dropna=False)['value']
        .agg(count='count', p95_value=lambda s: np.nanpercentile(s.dropna(),95) if s.dropna().size>0 else np.nan)
        .reset_index()
    )
    metric_thresholds_df['p95_value'] = metric_thresholds_df['p95_value'].round(6)

# Trace p95
try:
    trace_thresholds_df
except NameError:
    trace_full_candidates = trace_df[trace_df['cmdb_id'].isin(candidates)].copy()
    trace_thresholds_df = (
        trace_full_candidates
        .groupby(['cmdb_id','trace_name'], dropna=False)['value']
        .agg(count='count', p95_value=lambda s: np.nanpercentile(s.dropna(),95) if s.dropna().size>0 else np.nan)
        .reset_index()
    )
    trace_thresholds_df['p95_value'] = trace_thresholds_df['p95_value'].round(6)

# Log p95
try:
    log_thresholds_df
except NameError:
    log_full_candidates = log_df[log_df['cmdb_id'].isin(candidates)].copy()
    log_thresholds_df = (
        log_full_candidates
        .groupby(['cmdb_id','log_name'], dropna=False)['value']
        .agg(count='count', p95_value=lambda s: np.nanpercentile(s.dropna(),95) if s.dropna().size>0 else np.nan)
        .reset_index()
    )
    log_thresholds_df['p95_value'] = log_thresholds_df['p95_value'].round(6)

# Trace low-tail thresholds (P15 and P5)
try:
    trace_low_thresholds_df
except NameError:
    trace_full_candidates = trace_df[trace_df['cmdb_id'].isin(candidates)].copy()
    def p15(s): 
        s2 = s.dropna()
        return np.nanpercentile(s2,15) if s2.size>0 else np.nan
    def p5(s):
        s2 = s.dropna()
        return np.nanpercentile(s2,5) if s2.size>0 else np.nan
    trace_low_thresholds_df = (
        trace_full_candidates
        .groupby(['cmdb_id','trace_name'], dropna=False)['value']
        .agg(count='count', p15_value=lambda s: p15(s), p5_value=lambda s: p5(s))
        .reset_index()
    )
    trace_low_thresholds_df['p15_value'] = trace_low_thresholds_df['p15_value'].round(6)
    trace_low_thresholds_df['p5_value'] = trace_low_thresholds_df['p5_value'].round(6)

# Incident window 2021-03-09 10:00 to 10:30 UTC+8 (inclusive)
start_ts = tz.localize(pd.Timestamp('2021-03-09 10:00:00'))
end_ts   = tz.localize(pd.Timestamp('2021-03-09 10:30:00'))

# Filter windows and store as required kernel variables
metric_window = metric_df[
    metric_df['cmdb_id'].isin(candidates) &
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts)
].copy()

trace_window = trace_df[
    trace_df['cmdb_id'].isin(candidates) &
    (trace_df['timestamp'] >= start_ts) &
    (trace_df['timestamp'] <= end_ts)
].copy()

log_window = log_df[
    log_df['cmdb_id'].isin(candidates) &
    (log_df['timestamp'] >= start_ts) &
    (log_df['timestamp'] <= end_ts)
].copy()

# 1) Metric high-side anomalies using metric_thresholds_df
# Left join to obtain p95_value
metric_window = metric_window.merge(
    metric_thresholds_df[['cmdb_id','kpi_name','p95_value']],
    on=['cmdb_id','kpi_name'],
    how='left'
)

metric_window['is_anomaly'] = (~metric_window['p95_value'].isna()) & (~metric_window['value'].isna()) & (metric_window['value'] >= metric_window['p95_value'])

metric_anomaly_rows = metric_window[metric_window['is_anomaly']].copy()

if metric_anomaly_rows.shape[0] == 0:
    metric_anomalies_df = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','p95_value','severity_ratio'])
else:
    magg = metric_anomaly_rows.groupby(['cmdb_id','kpi_name'], dropna=False).agg(
        anomaly_count = ('is_anomaly','size'),
        earliest_anomaly_timestamp = ('timestamp','min'),
        max_value_in_window = ('value','max'),
        p95_value = ('p95_value','first')
    ).reset_index()
    def sev_m(r):
        p = r['p95_value']; mx = r['max_value_in_window']
        if pd.isna(p): return np.nan
        if p == 0: return np.inf
        return (mx - p)/p
    magg['severity_ratio'] = magg.apply(sev_m, axis=1)
    magg['max_value_in_window'] = magg['max_value_in_window'].round(6)
    magg['p95_value'] = magg['p95_value'].round(6)
    magg['severity_ratio'] = magg['severity_ratio'].replace([np.inf,-np.inf], np.inf)
    metric_anomalies_df = magg.sort_values(['anomaly_count','severity_ratio'], ascending=[False, False]).reset_index(drop=True)

# 2) Trace high-side anomalies using trace_thresholds_df
trace_window = trace_window.merge(
    trace_thresholds_df[['cmdb_id','trace_name','p95_value']],
    on=['cmdb_id','trace_name'],
    how='left'
)

trace_window['is_anomaly'] = (~trace_window['p95_value'].isna()) & (~trace_window['value'].isna()) & (trace_window['value'] >= trace_window['p95_value'])

trace_anomaly_rows = trace_window[trace_window['is_anomaly']].copy()

if trace_anomaly_rows.shape[0] == 0:
    trace_anomalies_df = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','p95_value','severity_ratio'])
else:
    tagg = trace_anomaly_rows.groupby(['cmdb_id','trace_name'], dropna=False).agg(
        anomaly_count = ('is_anomaly','size'),
        earliest_anomaly_timestamp = ('timestamp','min'),
        max_value_in_window = ('value','max'),
        p95_value = ('p95_value','first')
    ).reset_index()
    def sev_t(r):
        p = r['p95_value']; mx = r['max_value_in_window']
        if pd.isna(p): return np.nan
        if p == 0: return np.inf
        return (mx - p)/p
    tagg['severity_ratio'] = tagg.apply(sev_t, axis=1)
    tagg['max_value_in_window'] = tagg['max_value_in_window'].round(6)
    tagg['p95_value'] = tagg['p95_value'].round(6)
    tagg['severity_ratio'] = tagg['severity_ratio'].replace([np.inf,-np.inf], np.inf)
    trace_anomalies_df = tagg.sort_values(['anomaly_count','severity_ratio'], ascending=[False, False]).reset_index(drop=True)

# 3) Trace low-tail anomalies using trace_low_thresholds_df (p15/p5)
trace_window = trace_window.merge(
    trace_low_thresholds_df[['cmdb_id','trace_name','p15_value','p5_value']],
    on=['cmdb_id','trace_name'],
    how='left',
    suffixes=('','_low')
)

# Only consider rows with available p15 and p5
trace_window_low_valid = trace_window[~trace_window['p15_value'].isna() & ~trace_window['p5_value'].isna()].copy()

trace_window_low_valid['is_low_anomaly'] = (~trace_window_low_valid['value'].isna()) & (trace_window_low_valid['value'] <= trace_window_low_valid['p15_value'])
trace_window_low_valid['is_p5_or_below'] = (~trace_window_low_valid['value'].isna()) & (trace_window_low_valid['value'] <= trace_window_low_valid['p5_value'])

trace_low_rows = trace_window_low_valid[trace_window_low_valid['is_low_anomaly']].copy()

if trace_low_rows.shape[0] == 0:
    trace_low_anomalies_df = pd.DataFrame(columns=[
        'cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp','min_value_in_window','p15_value','p5_value','severity_ratio'
    ])
else:
    tlagg = trace_low_rows.groupby(['cmdb_id','trace_name'], dropna=False).agg(
        anomaly_count = ('is_low_anomaly','size'),
        earliest_anomaly_timestamp = ('timestamp','min'),
        min_value_in_window = ('value','min'),
        p15_value = ('p15_value','first'),
        p5_value = ('p5_value','first')
    ).reset_index()
    def sev_low(r):
        p = r['p15_value']; mn = r['min_value_in_window']
        if pd.isna(p): return np.nan
        if p == 0: return np.inf
        return (p - mn)/p
    tlagg['severity_ratio'] = tlagg.apply(sev_low, axis=1)
    tlagg['min_value_in_window'] = tlagg['min_value_in_window'].round(6)
    tlagg['p15_value'] = tlagg['p15_value'].round(6)
    tlagg['p5_value'] = tlagg['p5_value'].round(6)
    tlagg['severity_ratio'] = tlagg['severity_ratio'].replace([np.inf,-np.inf], np.inf)
    trace_low_anomalies_df = tlagg.sort_values(['anomaly_count','severity_ratio'], ascending=[False, False]).reset_index(drop=True)

# 4) Log high-side anomalies using log_thresholds_df
log_window = log_window.merge(
    log_thresholds_df[['cmdb_id','log_name','p95_value']],
    on=['cmdb_id','log_name'],
    how='left'
)

log_window['is_anomaly'] = (~log_window['p95_value'].isna()) & (~log_window['value'].isna()) & (log_window['value'] >= log_window['p95_value'])

log_anomaly_rows = log_window[log_window['is_anomaly']].copy()

if log_anomaly_rows.shape[0] == 0:
    log_anomalies_df = pd.DataFrame(columns=['cmdb_id','log_name','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','p95_value','severity_ratio'])
else:
    lagg = log_anomaly_rows.groupby(['cmdb_id','log_name'], dropna=False).agg(
        anomaly_count = ('is_anomaly','size'),
        earliest_anomaly_timestamp = ('timestamp','min'),
        max_value_in_window = ('value','max'),
        p95_value = ('p95_value','first')
    ).reset_index()
    def sev_l(r):
        p = r['p95_value']; mx = r['max_value_in_window']
        if pd.isna(p): return np.nan
        if p == 0: return np.inf
        return (mx - p)/p
    lagg['severity_ratio'] = lagg.apply(sev_l, axis=1)
    lagg['max_value_in_window'] = lagg['max_value_in_window'].round(6)
    lagg['p95_value'] = lagg['p95_value'].round(6)
    lagg['severity_ratio'] = lagg['severity_ratio'].replace([np.inf,-np.inf], np.inf)
    log_anomalies_df = lagg.sort_values(['anomaly_count','severity_ratio'], ascending=[False, False]).reset_index(drop=True)

# Prepare compact top-20 tables for display (selected columns)
metric_top20 = metric_anomalies_df[['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','p95_value','severity_ratio']].head(20)
trace_top20 = trace_anomalies_df[['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','p95_value','severity_ratio']].head(20)
trace_low_top20 = trace_low_anomalies_df[['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp','min_value_in_window','p15_value','p5_value','severity_ratio']].head(20)
log_top20 = log_anomalies_df[['cmdb_id','log_name','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','p95_value','severity_ratio']].head(20)

# Display the compact results (these variables are stored in kernel for later steps)
metric_top20, trace_top20, trace_low_top20, log_top20

```
Out[9]:
```
Summary of the full diagnosis for 2021-03-09 10:00–10:30 (UTC+8) using the provided candidate components and precomputed global thresholds:

General
- Time window: 2021-03-09 10:00–10:30 (inclusive, UTC+8).
- Global thresholds (P95, and P15/P5 for trace low-tail) were computed over the full available series before window filtering.
- The full aggregated anomaly tables are stored in kernel variables: metric_anomalies_df, trace_anomalies_df, trace_low_anomalies_df, log_anomalies_df.

Metric high-side anomalies (top results)
- Many metric groups are flagged with anomaly_count = 31 (one anomaly each minute across the full 31-minute window). Examples (top by anomaly_count / severity):
  - IG01 / OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT — anomaly_count 31, max=0.0, p95=0.0 → severity = Inf (p95==0).
  - IG02 / OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT — anomaly_count 31, max=0.0, p95=0.0 → severity = Inf.
  - Mysql01, Redis01, Redis02, Tomcat01/02 groups (several Tomcat session KPIs) also show anomaly_count=31 with p95_value == 0 and max_value == 0 → severity Inf.
- Interpretation: many metric anomalies are artifacts where the global P95 is 0 and observed values equal 0, producing infinite severity by the chosen severity formula. These should be reviewed to decide whether zero thresholds are meaningful or should be excluded.

Trace high-side anomalies (top 20; examples)
- Several groups with repeated anomalies (anomaly_count up to 3 in this window) and meaningful non-zero severities:
  - MG02 / trace.from_Tomcat04.row_count — anomaly_count 3, earliest 10:09, max=183, p95=174, severity ≈ 0.0517
  - Tomcat04 / trace.to_MG02.row_count — anomaly_count 3, earliest 10:09, max=183, p95=174, severity ≈ 0.0517
  - MG02 / trace.to_dockerB2.row_count — anomaly_count 3, earliest 10:01, max=143, p95=142, severity ≈ 0.0070
  - Tomcat04 / trace.self.duration_p95 — anomaly_count 2, earliest 10:05, max=1.004, p95≈0.126, severity ≈ 6.97 (large duration spike)
  - Tomcat02 / trace.self.duration_p95 — anomaly_count 2, earliest 10:05, max=1.005, p95≈0.368, severity ≈ 1.73
- Interpretation: trace anomalies include both modest row_count spikes and several substantial duration spikes (notably Tomcat04, Tomcat02 around 10:05).

Trace low-tail (drop) anomalies (top 20; examples)
- Some groups show repeated low-tail anomalies and notable severity (p15 - min)/p15:
  - MG01 / trace.from_dockerB1.duration_p95 — anomaly_count 8, earliest 10:01, min=0.008, p15=0.009, p5=0.008, severity ≈ 0.111
  - MG02 / trace.self.duration_p95 — anomaly_count 6, earliest 10:01, min=0.7467, p15≈1.0540, p5≈1.0370, severity ≈ 0.292
  - MG01 / trace.from_dockerA2.duration_mean — anomaly_count 6, earliest 10:02, min≈0.006227, p15≈0.0076, severity ≈ 0.181
  - Several row_count series for IG02/Tomcat02 and MG nodes show lower-than-usual counts with anomaly_count up to 6 and modest severity ratios.
- Interpretation: there are multiple low-tail groups, primarily affecting MG and some Tomcat/IG series; some show substantial relative drops.

Log high-side anomalies (top results)
- Many log.error_count groups are flagged across the entire window with anomaly_count up to 31, but p95_value == 0 and max_value == 0, producing severity = Inf:
  - Tomcat01 / log.error_count — anomaly_count 31, max=0.00, p95=0.00 → severity Inf
  - Tomcat03 / log.error_count — anomaly_count 31, ...
  - apache01 / log.error_count, apache02 / log.error_count, Tomcat04 / log.error_count, etc., same pattern.
- Notable non-inf results:
  - Tomcat01 / log.row_count — anomaly_count 3, earliest 10:02, max=366, p95=342, severity ≈ 0.070
  - Tomcat02 / log.gc_pause_max — anomaly_count 2, earliest 10:04, max=0.12, p95=0.099, severity ≈ 0.212
  - Several log.row_count groups for apache/Tomcat/Mysql have small positive severities.
- Interpretation: as with metrics, many log anomalies are due to p95 = 0; a few log series show meaningful increases (row_count, gc_pause_max).

Overall conclusions and recommendations
- The pipeline ran for 10:00–10:30 and found:
  - Trace high-side anomalies: notable duration spikes (Tomcat04, Tomcat02 around 10:05) and row_count spikes (MG02/Tomcat04 at ~10:09).
  - Trace low-tail anomalies: several MG/IG/Tomcat series show significant low-tail events (reduced durations / counts).
  - Metric and log results are dominated by groups where global P95 == 0 and observed values == 0; these produce infinite severity by the chosen formula and should be reviewed as possible threshold artifacts.
- Suggested next steps:
  - Inspect the trace duration spikes (Tomcat04, Tomcat02) at ~10:05 and the MG02/Tomcat04 row_count spike at ~10:09 for causal investigation.
  - Filter out series with zero-valued P95 (or handle them specially) to avoid misleading infinite severities for metrics/logs.
  - If you want, I can: (a) show the underlying trace time series (10:00–10:30) for the top anomalous trace groups, or (b) re-compute thresholds excluding long-zero series.

If you want a focused next action, tell me which anomaly group (e.g., "Tomcat04 / trace.self.duration_p95 at 10:05") to inspect further and I will extract the raw rows for that group in the 10:00–10:30 window.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window  p95_value  severity_ratio
0       IG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             31  2021-03-09 10:00:00+08:00                  0.0        0.0             inf
1       IG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             31  2021-03-09 10:00:00+08:00                  0.0        0.0             inf
2    Mysql01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             31  2021-03-09 10:00:00+08:00                  0.0        0.0             inf
3    Mysql01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31  2021-03-09 10:00:00+08:00                  0.0        0.0             inf
4    Redis01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             31  2021-03-09 10:00:00+08:00                  0.0        0.0             inf
5    Redis01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             31  2021-03-09 10:00:00+08:00                  0.0        0.0             inf
6    Redis02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31  2021-03-09 10:00:00+08:00                  0.0        0.0             inf
7   Tomcat01    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter             31  2021-03-09 10:00:00+08:00                  0.0        0.0             inf
8   Tomcat01  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...             31  2021-03-09 10:00:00+08:00                  0.0        0.0             inf
9   Tomcat01  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...             31  2021-03-09 10:00:00+08:00                  0.0        0.0             inf
10  Tomcat01        Tomcat-Sessions_7441--_SESSIONActiveCounter             31  2021-03-09 10:00:00+08:00                  0.0        0.0             inf
11  Tomcat01     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter             31  2021-03-09 10:00:00+08:00                  0.0        0.0             inf
12  Tomcat01     Tomcat-Sessions_7441--_SESSIONRejectedSessions             31  2021-03-09 10:00:00+08:00                  0.0        0.0             inf
13  Tomcat01  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31  2021-03-09 10:00:00+08:00                  0.0        0.0             inf
14  Tomcat01  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31  2021-03-09 10:00:00+08:00                  0.0        0.0             inf
15  Tomcat01  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31  2021-03-09 10:00:00+08:00                  0.0        0.0             inf
16  Tomcat02    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter             31  2021-03-09 10:00:00+08:00                  0.0        0.0             inf
17  Tomcat02  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...             31  2021-03-09 10:00:00+08:00                  0.0        0.0             inf
18  Tomcat02  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...             31  2021-03-09 10:00:00+08:00                  0.0        0.0             inf
19  Tomcat02        Tomcat-Sessions_7441--_SESSIONActiveCounter             31  2021-03-09 10:00:00+08:00                  0.0        0.0             inf,      cmdb_id                         trace_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window   p95_value  severity_ratio
0       MG02      trace.from_Tomcat04.row_count              3  2021-03-09 10:09:00+08:00           183.000000  174.000000        0.051724
1   Tomcat04            trace.to_MG02.row_count              3  2021-03-09 10:09:00+08:00           183.000000  174.000000        0.051724
2       MG02        trace.to_dockerB2.row_count              3  2021-03-09 10:01:00+08:00           143.000000  142.000000        0.007042
3   Tomcat04            trace.self.duration_p95              2  2021-03-09 10:05:00+08:00             1.004000    0.126042        6.965599
4   Tomcat02            trace.self.duration_p95              2  2021-03-09 10:05:00+08:00             1.005000    0.367767        1.732708
5       MG02   trace.from_Tomcat02.duration_p95              2  2021-03-09 10:05:00+08:00             3.644000    2.114580        0.723274
6   Tomcat02         trace.to_MG02.duration_p95              2  2021-03-09 10:05:00+08:00             3.644000    2.114580        0.723274
7   Tomcat03            trace.self.duration_p95              2  2021-03-09 10:05:00+08:00             1.004000    0.590652        0.699816
8       MG02  trace.from_dockerB2.duration_mean              2  2021-03-09 10:05:00+08:00             0.108400    0.065474        0.655619
9       IG02    trace.to_Tomcat03.duration_mean              2  2021-03-09 10:05:00+08:00             0.767000    0.468293        0.637863
10  Tomcat03      trace.from_IG02.duration_mean              2  2021-03-09 10:05:00+08:00             0.767000    0.468293        0.637863
11      MG02  trace.from_dockerA2.duration_mean              2  2021-03-09 10:03:00+08:00             0.216872    0.132977        0.630899
12      MG01  trace.from_dockerA2.duration_mean              2  2021-03-09 10:15:00+08:00             0.262875    0.162048        0.622205
13      MG02     trace.to_dockerA1.duration_p95              2  2021-03-09 10:05:00+08:00             1.769200    1.105195        0.600803
14      MG01    trace.to_dockerA2.duration_mean              2  2021-03-09 10:05:00+08:00             0.848807    0.551770        0.538335
15      MG02   trace.from_dockerB2.duration_p95              2  2021-03-09 10:05:00+08:00             1.014300    0.667230        0.520165
16      MG01  trace.from_Tomcat03.duration_mean              2  2021-03-09 10:05:00+08:00             0.715919    0.476497        0.502463
17  Tomcat03        trace.to_MG01.duration_mean              2  2021-03-09 10:05:00+08:00             0.715919    0.476497        0.502463
18      MG01    trace.to_dockerA1.duration_mean              2  2021-03-09 10:00:00+08:00             0.415804    0.277249        0.499749
19      IG02     trace.to_Tomcat03.duration_p95              2  2021-03-09 10:05:00+08:00             3.144400    2.118280        0.484412,      cmdb_id                         trace_name  anomaly_count earliest_anomaly_timestamp  min_value_in_window   p15_value    p5_value  severity_ratio
0       MG01   trace.from_dockerB1.duration_p95              8  2021-03-09 10:01:00+08:00             0.008000    0.009000    0.008000        0.111111
1       MG02            trace.self.duration_p95              6  2021-03-09 10:01:00+08:00             0.746700    1.054000    1.037005        0.291556
2       MG01  trace.from_dockerA2.duration_mean              6  2021-03-09 10:02:00+08:00             0.006227    0.007600    0.006622        0.180658
3       IG02        trace.to_Tomcat02.row_count              6  2021-03-09 10:05:00+08:00            93.000000  112.000000   97.150000        0.169643
4   Tomcat02          trace.from_IG02.row_count              6  2021-03-09 10:05:00+08:00            93.000000  112.000000   97.150000        0.169643
5   Tomcat03            trace.self.duration_p95              6  2021-03-09 10:01:00+08:00             0.031800    0.037000    0.034752        0.140541
6       MG02        trace.to_dockerB2.row_count              6  2021-03-09 10:02:00+08:00            89.000000   99.000000   92.250000        0.101010
7       MG02   trace.from_dockerB2.duration_p95              6  2021-03-09 10:02:00+08:00             0.007350    0.008000    0.008000        0.081250
8       MG02      trace.from_Tomcat01.row_count              6  2021-03-09 10:06:00+08:00            99.000000  107.000000  104.900000        0.074766
9   Tomcat01            trace.to_MG02.row_count              6  2021-03-09 10:06:00+08:00            99.000000  107.000000  104.900000        0.074766
10      MG02   trace.from_dockerA1.duration_p95              6  2021-03-09 10:01:00+08:00             0.008000    0.008000    0.008000        0.000000
11      MG01     trace.to_dockerA1.duration_p95              5  2021-03-09 10:02:00+08:00             0.036000    1.030140    1.027225        0.965053
12      MG02     trace.to_dockerB2.duration_p95              5  2021-03-09 10:01:00+08:00             0.090700    1.039155    0.663488        0.912718
13      MG01     trace.to_dockerB2.duration_p95              5  2021-03-09 10:01:00+08:00             0.144200    1.018212    0.987335        0.858379
14      MG02  trace.from_dockerB1.duration_mean              5  2021-03-09 10:10:00+08:00             0.006400    0.030090    0.011213        0.787305
15      MG02      trace.from_dockerA2.row_count              5  2021-03-09 10:15:00+08:00             4.000000   16.050000   10.250000        0.750779
16      MG01  trace.from_dockerB1.duration_mean              5  2021-03-09 10:12:00+08:00             0.006237    0.021806    0.018384        0.713978
17      MG01      trace.from_dockerA2.row_count              5  2021-03-09 10:15:00+08:00             5.000000   14.150000    8.900000        0.646643
18      MG01  trace.from_dockerB2.duration_mean              5  2021-03-09 10:08:00+08:00             0.009395    0.025414    0.019177        0.630322
19      MG02  trace.from_dockerB2.duration_mean              5  2021-03-09 10:06:00+08:00             0.008118    0.020597    0.010479        0.605865,      cmdb_id          log_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window  p95_value  severity_ratio
0   Tomcat01   log.error_count             31  2021-03-09 10:00:00+08:00                 0.00     0.0000             inf
1   Tomcat03   log.error_count             31  2021-03-09 10:00:00+08:00                 0.00     0.0000             inf
2   Tomcat04   log.error_count             31  2021-03-09 10:00:00+08:00                 0.00     0.0000             inf
3   apache01   log.error_count             31  2021-03-09 10:00:00+08:00                 0.00     0.0000             inf
4   apache02   log.error_count             31  2021-03-09 10:00:00+08:00                 0.00     0.0000             inf
5    Mysql01   log.error_count             29  2021-03-09 10:00:00+08:00                 0.00     0.0000             inf
6   Tomcat02   log.error_count             23  2021-03-09 10:00:00+08:00                 0.00     0.0000             inf
7   Tomcat02     log.row_count             23  2021-03-09 10:00:00+08:00                 1.00     1.0000        0.000000
8   Tomcat01     log.row_count              3  2021-03-09 10:02:00+08:00               366.00   342.0000        0.070175
9   Tomcat02  log.gc_pause_max              2  2021-03-09 10:04:00+08:00                 0.12     0.0990        0.212121
10   Mysql01     log.row_count              2  2021-03-09 10:05:00+08:00                 9.00     8.2000        0.097561
11  Tomcat03     log.row_count              2  2021-03-09 10:16:00+08:00               343.00   318.0000        0.078616
12  apache02     log.row_count              2  2021-03-09 10:16:00+08:00               692.00   646.0000        0.071207
13  Tomcat04     log.row_count              2  2021-03-09 10:09:00+08:00               347.00   334.0000        0.038922
14  apache01     log.row_count              2  2021-03-09 10:16:00+08:00               674.00   654.5000        0.029794
15  Tomcat03  log.gc_pause_max              2  2021-03-09 10:04:00+08:00                 0.08     0.0800        0.000000
16  Tomcat04  log.gc_pause_max              1  2021-03-09 10:05:00+08:00                 0.08     0.0735        0.088435
17  Tomcat01  log.gc_pause_max              1  2021-03-09 10:03:00+08:00                 0.09     0.0835        0.077844)```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing dataframes if present; otherwise load from CSV files
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
    if 'timestamp' in metric_df.columns:
        metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', errors='coerce').dt.tz_localize(tz)

try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')
    if 'timestamp' in trace_df.columns:
        trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', errors='coerce').dt.tz_localize(tz)

try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')
    if 'timestamp' in log_df.columns:
        log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', errors='coerce').dt.tz_localize(tz)

# Target component
component = 'Tomcat04'

# 1) metric.csv: distinct kpi_name values with counts for Tomcat04
if ('cmdb_id' in metric_df.columns) and ('kpi_name' in metric_df.columns):
    tomcat04_metric_kpis = (
        metric_df[metric_df['cmdb_id'] == component]
        .groupby('kpi_name', dropna=False)
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .head(50)
        .reset_index(drop=True)
    )
else:
    tomcat04_metric_kpis = pd.DataFrame(columns=['kpi_name', 'count'])

# 2) trace.csv: distinct trace_name values with counts for Tomcat04
if ('cmdb_id' in trace_df.columns) and ('trace_name' in trace_df.columns):
    tomcat04_trace_names = (
        trace_df[trace_df['cmdb_id'] == component]
        .groupby('trace_name', dropna=False)
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .head(50)
        .reset_index(drop=True)
    )
else:
    tomcat04_trace_names = pd.DataFrame(columns=['trace_name', 'count'])

# 3) log.csv: distinct log_name values with counts for Tomcat04
if ('cmdb_id' in log_df.columns) and ('log_name' in log_df.columns):
    tomcat04_log_names = (
        log_df[log_df['cmdb_id'] == component]
        .groupby('log_name', dropna=False)
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .head(20)
        .reset_index(drop=True)
    )
else:
    tomcat04_log_names = pd.DataFrame(columns=['log_name', 'count'])

# Display only the requested aggregated tables (kernel will show these variables)
tomcat04_metric_kpis, tomcat04_trace_names, tomcat04_log_names

```
Out[10]:
```
# Compose a plain-English summary for Tomcat04 using the aggregated tables computed earlier.
import pandas as pd

metric_df = globals().get('tomcat04_metric_kpis', pd.DataFrame(columns=['kpi_name','count']))
trace_df  = globals().get('tomcat04_trace_names', pd.DataFrame(columns=['trace_name','count']))
log_df    = globals().get('tomcat04_log_names', pd.DataFrame(columns=['log_name','count']))

n_metric = int(metric_df.shape[0])
n_trace  = int(trace_df.shape[0])
n_log    = int(log_df.shape[0])

def top_list(df, key_col, n=5):
    if df.empty:
        return ["<none>"]
    return [f"{row[key_col]} (count={int(row['count'])})" for _, row in df.head(n).iterrows()]

metric_top5 = top_list(metric_df, 'kpi_name', 5)
trace_top5  = top_list(trace_df, 'trace_name', 10)  # show more for trace
log_top5    = top_list(log_df, 'log_name', 5)

summary = (
    "Tomcat04 telemetry keys summary:\n\n"
    f"metric.csv:\n"
    f" - distinct kpi_name rows returned: {n_metric} (list limited to 50 as requested)\n"
    f" - top examples: \n    - " + "\n    - ".join(metric_top5) + "\n\n"
    f"trace.csv:\n"
    f" - distinct trace_name rows returned: {n_trace} (list limited to 50)\n"
    f" - top examples: \n    - " + "\n    - ".join(trace_top5) + "\n\n"
    f"log.csv:\n"
    f" - distinct log_name rows returned: {n_log} (list limited to 20)\n"
    f" - items: \n    - " + "\n    - ".join(log_top5) + "\n\n"
    "Notes:\n"
    "- The metric list was truncated to 50 rows when requested; there may be additional KPI names beyond those returned.\n"
    "- Use these keys (examples above) to target follow-up inspections (e.g., trace.self.duration_p95, trace.from_IG01.duration_mean, Tomcat session KPIs, log.error_count, log.row_count).\n"
    "- Full aggregated tables are stored in kernel variables: tomcat04_metric_kpis, tomcat04_trace_names, tomcat04_log_names."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(                                             kpi_name  count
0   Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31
1   Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31
2   Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31
3   Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31
4   Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31
..                                                ...    ...
45                    OSLinux-CPU_CPU-2_SingleCpuUtil     10
46                    OSLinux-CPU_CPU-1_SingleCpuidle     10
47                    OSLinux-CPU_CPU-3_SingleCpuUtil     10
48                    OSLinux-CPU_CPU-2_SingleCpuidle     10
49     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps     10

[50 rows x 2 columns],                        trace_name  count
0   trace.from_IG01.duration_mean     30
1    trace.from_IG01.duration_p95     30
2       trace.from_IG01.row_count     30
3   trace.from_IG02.duration_mean     30
4    trace.from_IG02.duration_p95     30
5       trace.from_IG02.row_count     30
6        trace.self.duration_mean     30
7     trace.to_MG01.duration_mean     30
8            trace.self.row_count     30
9         trace.self.duration_p95     30
10        trace.to_MG01.row_count     30
11    trace.to_MG02.duration_mean     30
12     trace.to_MG02.duration_p95     30
13     trace.to_MG01.duration_p95     30
14        trace.to_MG02.row_count     30
15      trace.from_root.row_count      5
16  trace.from_root.duration_mean      5
17   trace.from_root.duration_p95      5,            log_name  count
0   log.error_count     31
1     log.row_count     31
2  log.gc_pause_max     14)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[11]:

import pandas as pd
import numpy as np
import pytz

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Reuse existing trace_df if present, otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

# Ensure timestamp column is timezone-aware in Asia/Shanghai
if 'timestamp' in trace_df.columns:
    col = trace_df['timestamp']
    if pd.api.types.is_integer_dtype(col) or pd.api.types.is_float_dtype(col):
        trace_df['timestamp'] = pd.to_datetime(col, unit='s', errors='coerce').dt.tz_localize(tz)
    else:
        coerced = pd.to_datetime(col, errors='coerce')
        if coerced.dt.tz is None:
            trace_df['timestamp'] = coerced.dt.tz_localize(tz)
        else:
            trace_df['timestamp'] = coerced.dt.tz_convert(tz)

# Ensure numeric value column
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Incident window (inclusive)
start_ts = tz.localize(pd.Timestamp('2021-03-09 10:00:00'))
end_ts   = tz.localize(pd.Timestamp('2021-03-09 10:30:00'))

# Helper to extract series for a (cmdb_id, trace_name) pair
def extract_series(df, cmdb_id, trace_name, start=start_ts, end=end_ts, limit=20):
    cols = ['timestamp','cmdb_id','trace_name','value']
    if not {'timestamp','cmdb_id','trace_name','value'}.issubset(df.columns):
        return pd.DataFrame(columns=cols)
    s = df[
        (df['cmdb_id'] == cmdb_id) &
        (df['trace_name'] == trace_name) &
        (df['timestamp'] >= start) &
        (df['timestamp'] <= end)
    ].copy().sort_values('timestamp', ascending=True)
    # keep only requested columns and at most `limit` rows for display
    out = s[cols].head(limit).reset_index(drop=True)
    return out

# Define requested groups and corresponding variable names
groups = [
    ('Tomcat04','trace.self.duration_p95','tomcat04_self_duration_p95_series'),
    ('Tomcat04','trace.self.duration_mean','tomcat04_self_duration_mean_series'),
    ('Tomcat04','trace.to_MG02.row_count','tomcat04_to_MG02_rowcount_series'),
    ('MG02','trace.from_Tomcat04.row_count','mg02_from_tomcat04_rowcount_series'),
    ('Tomcat02','trace.self.duration_p95','tomcat02_self_duration_p95_series'),
    ('Tomcat02','trace.to_MG02.duration_p95','tomcat02_to_MG02_duration_p95_series'),
    ('MG02','trace.from_Tomcat02.duration_p95','mg02_from_tomcat02_duration_p95_series'),
]

# Extract series and collect summary metrics
summary_rows = []
# Prepare variables in local namespace
for cmdb, tname, varname in groups:
    series_df = extract_series(trace_df, cmdb, tname)
    # store in kernel variable by name
    globals()[varname] = series_df
    # compute summary metrics for the full window (not limited to 20)
    full_df = trace_df[
        (trace_df['cmdb_id'] == cmdb) &
        (trace_df['trace_name'] == tname) &
        (trace_df['timestamp'] >= start_ts) &
        (trace_df['timestamp'] <= end_ts)
    ].copy()
    if full_df.empty:
        row_count = 0
        earliest = None
        latest = None
        min_v = np.nan
        max_v = np.nan
        mean_v = np.nan
    else:
        row_count = int(full_df.shape[0])
        earliest = full_df['timestamp'].min().isoformat()
        latest = full_df['timestamp'].max().isoformat()
        min_v = float(full_df['value'].min())
        max_v = float(full_df['value'].max())
        mean_v = float(full_df['value'].mean())
    summary_rows.append({
        'cmdb_id': cmdb,
        'trace_name': tname,
        'row_count_in_window': row_count,
        'earliest_timestamp': earliest,
        'latest_timestamp': latest,
        'min_value': round(min_v,6) if not np.isnan(min_v) else np.nan,
        'max_value': round(max_v,6) if not np.isnan(max_v) else np.nan,
        'mean_value': round(mean_v,6) if not np.isnan(mean_v) else np.nan
    })

# Create compact summary DataFrame limited to these groups
groups_summary = pd.DataFrame(summary_rows)

# Display only the per-group extracted tables (each limited to 20 rows) and the compact summary
tomcat04_self_duration_p95_series, tomcat04_self_duration_mean_series, tomcat04_to_MG02_rowcount_series, mg02_from_tomcat04_rowcount_series, tomcat02_self_duration_p95_series, tomcat02_to_MG02_duration_p95_series, mg02_from_tomcat02_duration_p95_series, groups_summary

```
Out[11]:
```
Summary of extracted trace series for 2021-03-09 10:00–10:30 (UTC+8):

Groups & window statistics (row_count, earliest → latest, min, max, mean)

- Tomcat04 / trace.self.duration_p95
  - rows: 30, 2021-03-09T10:00:00+08:00 → 2021-03-09T10:29:00+08:00
  - min = 0.033100, max = 1.004000, mean = 0.083338
  - Note: a large spike to 1.004 at 10:15.

- Tomcat04 / trace.self.duration_mean
  - rows: 30, 2021-03-09T10:00:00+08:00 → 2021-03-09T10:29:00+08:00
  - min = 0.028898, max = 0.086868, mean = 0.047264

- Tomcat04 / trace.to_MG02.row_count
  - rows: 30, 2021-03-09T10:00:00+08:00 → 2021-03-09T10:29:00+08:00
  - min = 98.0, max = 183.0, mean = 135.4
  - Note: peak 183 observed at 10:16.

- MG02 / trace.from_Tomcat04.row_count
  - rows: 30, 2021-03-09T10:00:00+08:00 → 2021-03-09T10:29:00+08:00
  - min = 98.0, max = 183.0, mean = 135.4
  - (mirrors Tomcat04 → MG02 row_count; peak at 10:16)

- Tomcat02 / trace.self.duration_p95
  - rows: 30, 2021-03-09T10:00:00+08:00 → 2021-03-09T10:29:00+08:00
  - min = 0.035000, max = 1.005000, mean = 0.100363
  - Note: spike to 1.005 at 10:05.

- Tomcat02 / trace.to_MG02.duration_p95
  - rows: 30, 2021-03-09T10:00:00+08:00 → 2021-03-09T10:29:00+08:00
  - min = 0.543200, max = 3.644000, mean = 1.374552
  - Note: large spike to 3.644 at 10:05.

- MG02 / trace.from_Tomcat02.duration_p95
  - rows: 30, 2021-03-09T10:00:00+08:00 → 2021-03-09T10:29:00+08:00
  - min = 0.543200, max = 3.644000, mean = 1.374552
  - (mirrors Tomcat02 → MG02 durations; spike at 10:05)

Stored kernel variables (per your request)
- tomcat04_self_duration_p95_series
- tomcat04_self_duration_mean_series
- tomcat04_to_MG02_rowcount_series
- mg02_from_tomcat04_rowcount_series
- tomcat02_self_duration_p95_series
- tomcat02_to_MG02_duration_p95_series
- mg02_from_tomcat02_duration_p95_series

Suggested next steps
- Investigate the duration spikes:
  - Tomcat02 → MG02 duration spike at 10:05 (value 3.644).
  - Tomcat02 self.duration_p95 spike at 10:05 (1.005).
  - Tomcat04 self.duration_p95 spike at 10:15 (1.004).
- Investigate the row_count peak between Tomcat04 and MG02 around 10:16 (183).
If you want, I can now (for any one selected group) provide the full per-minute rows for the 10:00–10:30 window, or show neighboring metrics/logs around the spike times.

The original code execution output of IPython Kernel is also provided below for reference:

(                   timestamp   cmdb_id               trace_name    value
0  2021-03-09 10:00:00+08:00  Tomcat04  trace.self.duration_p95  0.04510
1  2021-03-09 10:01:00+08:00  Tomcat04  trace.self.duration_p95  0.03500
2  2021-03-09 10:02:00+08:00  Tomcat04  trace.self.duration_p95  0.03700
3  2021-03-09 10:03:00+08:00  Tomcat04  trace.self.duration_p95  0.05300
4  2021-03-09 10:04:00+08:00  Tomcat04  trace.self.duration_p95  0.07500
5  2021-03-09 10:05:00+08:00  Tomcat04  trace.self.duration_p95  0.12935
6  2021-03-09 10:06:00+08:00  Tomcat04  trace.self.duration_p95  0.05150
7  2021-03-09 10:07:00+08:00  Tomcat04  trace.self.duration_p95  0.07000
8  2021-03-09 10:08:00+08:00  Tomcat04  trace.self.duration_p95  0.04000
9  2021-03-09 10:09:00+08:00  Tomcat04  trace.self.duration_p95  0.05000
10 2021-03-09 10:10:00+08:00  Tomcat04  trace.self.duration_p95  0.04400
11 2021-03-09 10:11:00+08:00  Tomcat04  trace.self.duration_p95  0.04200
12 2021-03-09 10:12:00+08:00  Tomcat04  trace.self.duration_p95  0.04000
13 2021-03-09 10:13:00+08:00  Tomcat04  trace.self.duration_p95  0.03310
14 2021-03-09 10:14:00+08:00  Tomcat04  trace.self.duration_p95  0.05790
15 2021-03-09 10:15:00+08:00  Tomcat04  trace.self.duration_p95  1.00400
16 2021-03-09 10:16:00+08:00  Tomcat04  trace.self.duration_p95  0.12200
17 2021-03-09 10:17:00+08:00  Tomcat04  trace.self.duration_p95  0.04600
18 2021-03-09 10:18:00+08:00  Tomcat04  trace.self.duration_p95  0.04000
19 2021-03-09 10:19:00+08:00  Tomcat04  trace.self.duration_p95  0.04670,                    timestamp   cmdb_id                trace_name     value
0  2021-03-09 10:00:00+08:00  Tomcat04  trace.self.duration_mean  0.052407
1  2021-03-09 10:01:00+08:00  Tomcat04  trace.self.duration_mean  0.032192
2  2021-03-09 10:02:00+08:00  Tomcat04  trace.self.duration_mean  0.028898
3  2021-03-09 10:03:00+08:00  Tomcat04  trace.self.duration_mean  0.055317
4  2021-03-09 10:04:00+08:00  Tomcat04  trace.self.duration_mean  0.057218
5  2021-03-09 10:05:00+08:00  Tomcat04  trace.self.duration_mean  0.083749
6  2021-03-09 10:06:00+08:00  Tomcat04  trace.self.duration_mean  0.042891
7  2021-03-09 10:07:00+08:00  Tomcat04  trace.self.duration_mean  0.062977
8  2021-03-09 10:08:00+08:00  Tomcat04  trace.self.duration_mean  0.039428
9  2021-03-09 10:09:00+08:00  Tomcat04  trace.self.duration_mean  0.043709
10 2021-03-09 10:10:00+08:00  Tomcat04  trace.self.duration_mean  0.039070
11 2021-03-09 10:11:00+08:00  Tomcat04  trace.self.duration_mean  0.032849
12 2021-03-09 10:12:00+08:00  Tomcat04  trace.self.duration_mean  0.031050
13 2021-03-09 10:13:00+08:00  Tomcat04  trace.self.duration_mean  0.035977
14 2021-03-09 10:14:00+08:00  Tomcat04  trace.self.duration_mean  0.057259
15 2021-03-09 10:15:00+08:00  Tomcat04  trace.self.duration_mean  0.086868
16 2021-03-09 10:16:00+08:00  Tomcat04  trace.self.duration_mean  0.075902
17 2021-03-09 10:17:00+08:00  Tomcat04  trace.self.duration_mean  0.060678
18 2021-03-09 10:18:00+08:00  Tomcat04  trace.self.duration_mean  0.032305
19 2021-03-09 10:19:00+08:00  Tomcat04  trace.self.duration_mean  0.051596,                    timestamp   cmdb_id               trace_name  value
0  2021-03-09 10:00:00+08:00  Tomcat04  trace.to_MG02.row_count  146.0
1  2021-03-09 10:01:00+08:00  Tomcat04  trace.to_MG02.row_count  151.0
2  2021-03-09 10:02:00+08:00  Tomcat04  trace.to_MG02.row_count  125.0
3  2021-03-09 10:03:00+08:00  Tomcat04  trace.to_MG02.row_count  155.0
4  2021-03-09 10:04:00+08:00  Tomcat04  trace.to_MG02.row_count  135.0
5  2021-03-09 10:05:00+08:00  Tomcat04  trace.to_MG02.row_count  121.0
6  2021-03-09 10:06:00+08:00  Tomcat04  trace.to_MG02.row_count  127.0
7  2021-03-09 10:07:00+08:00  Tomcat04  trace.to_MG02.row_count  130.0
8  2021-03-09 10:08:00+08:00  Tomcat04  trace.to_MG02.row_count  158.0
9  2021-03-09 10:09:00+08:00  Tomcat04  trace.to_MG02.row_count  174.0
10 2021-03-09 10:10:00+08:00  Tomcat04  trace.to_MG02.row_count  129.0
11 2021-03-09 10:11:00+08:00  Tomcat04  trace.to_MG02.row_count  134.0
12 2021-03-09 10:12:00+08:00  Tomcat04  trace.to_MG02.row_count  120.0
13 2021-03-09 10:13:00+08:00  Tomcat04  trace.to_MG02.row_count  151.0
14 2021-03-09 10:14:00+08:00  Tomcat04  trace.to_MG02.row_count  153.0
15 2021-03-09 10:15:00+08:00  Tomcat04  trace.to_MG02.row_count  129.0
16 2021-03-09 10:16:00+08:00  Tomcat04  trace.to_MG02.row_count  183.0
17 2021-03-09 10:17:00+08:00  Tomcat04  trace.to_MG02.row_count  174.0
18 2021-03-09 10:18:00+08:00  Tomcat04  trace.to_MG02.row_count  171.0
19 2021-03-09 10:19:00+08:00  Tomcat04  trace.to_MG02.row_count  132.0,                    timestamp cmdb_id                     trace_name  value
0  2021-03-09 10:00:00+08:00    MG02  trace.from_Tomcat04.row_count  146.0
1  2021-03-09 10:01:00+08:00    MG02  trace.from_Tomcat04.row_count  151.0
2  2021-03-09 10:02:00+08:00    MG02  trace.from_Tomcat04.row_count  125.0
3  2021-03-09 10:03:00+08:00    MG02  trace.from_Tomcat04.row_count  155.0
4  2021-03-09 10:04:00+08:00    MG02  trace.from_Tomcat04.row_count  135.0
5  2021-03-09 10:05:00+08:00    MG02  trace.from_Tomcat04.row_count  121.0
6  2021-03-09 10:06:00+08:00    MG02  trace.from_Tomcat04.row_count  127.0
7  2021-03-09 10:07:00+08:00    MG02  trace.from_Tomcat04.row_count  130.0
8  2021-03-09 10:08:00+08:00    MG02  trace.from_Tomcat04.row_count  158.0
9  2021-03-09 10:09:00+08:00    MG02  trace.from_Tomcat04.row_count  174.0
10 2021-03-09 10:10:00+08:00    MG02  trace.from_Tomcat04.row_count  129.0
11 2021-03-09 10:11:00+08:00    MG02  trace.from_Tomcat04.row_count  134.0
12 2021-03-09 10:12:00+08:00    MG02  trace.from_Tomcat04.row_count  120.0
13 2021-03-09 10:13:00+08:00    MG02  trace.from_Tomcat04.row_count  151.0
14 2021-03-09 10:14:00+08:00    MG02  trace.from_Tomcat04.row_count  153.0
15 2021-03-09 10:15:00+08:00    MG02  trace.from_Tomcat04.row_count  129.0
16 2021-03-09 10:16:00+08:00    MG02  trace.from_Tomcat04.row_count  183.0
17 2021-03-09 10:17:00+08:00    MG02  trace.from_Tomcat04.row_count  174.0
18 2021-03-09 10:18:00+08:00    MG02  trace.from_Tomcat04.row_count  171.0
19 2021-03-09 10:19:00+08:00    MG02  trace.from_Tomcat04.row_count  132.0,                    timestamp   cmdb_id               trace_name    value
0  2021-03-09 10:00:00+08:00  Tomcat02  trace.self.duration_p95  0.06950
1  2021-03-09 10:01:00+08:00  Tomcat02  trace.self.duration_p95  0.03800
2  2021-03-09 10:02:00+08:00  Tomcat02  trace.self.duration_p95  0.03500
3  2021-03-09 10:03:00+08:00  Tomcat02  trace.self.duration_p95  0.04200
4  2021-03-09 10:04:00+08:00  Tomcat02  trace.self.duration_p95  0.05100
5  2021-03-09 10:05:00+08:00  Tomcat02  trace.self.duration_p95  1.00500
6  2021-03-09 10:06:00+08:00  Tomcat02  trace.self.duration_p95  0.05230
7  2021-03-09 10:07:00+08:00  Tomcat02  trace.self.duration_p95  0.08480
8  2021-03-09 10:08:00+08:00  Tomcat02  trace.self.duration_p95  0.04500
9  2021-03-09 10:09:00+08:00  Tomcat02  trace.self.duration_p95  0.04600
10 2021-03-09 10:10:00+08:00  Tomcat02  trace.self.duration_p95  0.04300
11 2021-03-09 10:11:00+08:00  Tomcat02  trace.self.duration_p95  0.03600
12 2021-03-09 10:12:00+08:00  Tomcat02  trace.self.duration_p95  0.03700
13 2021-03-09 10:13:00+08:00  Tomcat02  trace.self.duration_p95  0.04455
14 2021-03-09 10:14:00+08:00  Tomcat02  trace.self.duration_p95  0.10990
15 2021-03-09 10:15:00+08:00  Tomcat02  trace.self.duration_p95  0.57875
16 2021-03-09 10:16:00+08:00  Tomcat02  trace.self.duration_p95  0.04865
17 2021-03-09 10:17:00+08:00  Tomcat02  trace.self.duration_p95  0.08035
18 2021-03-09 10:18:00+08:00  Tomcat02  trace.self.duration_p95  0.04370
19 2021-03-09 10:19:00+08:00  Tomcat02  trace.self.duration_p95  0.04210,                    timestamp   cmdb_id                  trace_name    value
0  2021-03-09 10:00:00+08:00  Tomcat02  trace.to_MG02.duration_p95  1.11050
1  2021-03-09 10:01:00+08:00  Tomcat02  trace.to_MG02.duration_p95  1.07900
2  2021-03-09 10:02:00+08:00  Tomcat02  trace.to_MG02.duration_p95  1.04405
3  2021-03-09 10:03:00+08:00  Tomcat02  trace.to_MG02.duration_p95  1.08540
4  2021-03-09 10:04:00+08:00  Tomcat02  trace.to_MG02.duration_p95  1.08370
5  2021-03-09 10:05:00+08:00  Tomcat02  trace.to_MG02.duration_p95  3.64400
6  2021-03-09 10:06:00+08:00  Tomcat02  trace.to_MG02.duration_p95  1.89235
7  2021-03-09 10:07:00+08:00  Tomcat02  trace.to_MG02.duration_p95  2.09840
8  2021-03-09 10:08:00+08:00  Tomcat02  trace.to_MG02.duration_p95  1.08620
9  2021-03-09 10:09:00+08:00  Tomcat02  trace.to_MG02.duration_p95  1.08320
10 2021-03-09 10:10:00+08:00  Tomcat02  trace.to_MG02.duration_p95  1.05720
11 2021-03-09 10:11:00+08:00  Tomcat02  trace.to_MG02.duration_p95  1.06030
12 2021-03-09 10:12:00+08:00  Tomcat02  trace.to_MG02.duration_p95  1.08900
13 2021-03-09 10:13:00+08:00  Tomcat02  trace.to_MG02.duration_p95  2.07060
14 2021-03-09 10:14:00+08:00  Tomcat02  trace.to_MG02.duration_p95  2.07140
15 2021-03-09 10:15:00+08:00  Tomcat02  trace.to_MG02.duration_p95  2.12160
16 2021-03-09 10:16:00+08:00  Tomcat02  trace.to_MG02.duration_p95  2.10600
17 2021-03-09 10:17:00+08:00  Tomcat02  trace.to_MG02.duration_p95  1.56600
18 2021-03-09 10:18:00+08:00  Tomcat02  trace.to_MG02.duration_p95  1.06880
19 2021-03-09 10:19:00+08:00  Tomcat02  trace.to_MG02.duration_p95  1.08205,                    timestamp cmdb_id                        trace_name    value
0  2021-03-09 10:00:00+08:00    MG02  trace.from_Tomcat02.duration_p95  1.11050
1  2021-03-09 10:01:00+08:00    MG02  trace.from_Tomcat02.duration_p95  1.07900
2  2021-03-09 10:02:00+08:00    MG02  trace.from_Tomcat02.duration_p95  1.04405
3  2021-03-09 10:03:00+08:00    MG02  trace.from_Tomcat02.duration_p95  1.08540
4  2021-03-09 10:04:00+08:00    MG02  trace.from_Tomcat02.duration_p95  1.08370
5  2021-03-09 10:05:00+08:00    MG02  trace.from_Tomcat02.duration_p95  3.64400
6  2021-03-09 10:06:00+08:00    MG02  trace.from_Tomcat02.duration_p95  1.89235
7  2021-03-09 10:07:00+08:00    MG02  trace.from_Tomcat02.duration_p95  2.09840
8  2021-03-09 10:08:00+08:00    MG02  trace.from_Tomcat02.duration_p95  1.08620
9  2021-03-09 10:09:00+08:00    MG02  trace.from_Tomcat02.duration_p95  1.08320
10 2021-03-09 10:10:00+08:00    MG02  trace.from_Tomcat02.duration_p95  1.05720
11 2021-03-09 10:11:00+08:00    MG02  trace.from_Tomcat02.duration_p95  1.06030
12 2021-03-09 10:12:00+08:00    MG02  trace.from_Tomcat02.duration_p95  1.08900
13 2021-03-09 10:13:00+08:00    MG02  trace.from_Tomcat02.duration_p95  2.07060
14 2021-03-09 10:14:00+08:00    MG02  trace.from_Tomcat02.duration_p95  2.07140
15 2021-03-09 10:15:00+08:00    MG02  trace.from_Tomcat02.duration_p95  2.12160
16 2021-03-09 10:16:00+08:00    MG02  trace.from_Tomcat02.duration_p95  2.10600
17 2021-03-09 10:17:00+08:00    MG02  trace.from_Tomcat02.duration_p95  1.56600
18 2021-03-09 10:18:00+08:00    MG02  trace.from_Tomcat02.duration_p95  1.06880
19 2021-03-09 10:19:00+08:00    MG02  trace.from_Tomcat02.duration_p95  1.08205,     cmdb_id                        trace_name  row_count_in_window         earliest_timestamp           latest_timestamp  min_value   max_value  mean_value
0  Tomcat04           trace.self.duration_p95                   30  2021-03-09T10:00:00+08:00  2021-03-09T10:29:00+08:00   0.033100    1.004000    0.083338
1  Tomcat04          trace.self.duration_mean                   30  2021-03-09T10:00:00+08:00  2021-03-09T10:29:00+08:00   0.028898    0.086868    0.047264
2  Tomcat04           trace.to_MG02.row_count                   30  2021-03-09T10:00:00+08:00  2021-03-09T10:29:00+08:00  98.000000  183.000000  135.400000
3      MG02     trace.from_Tomcat04.row_count                   30  2021-03-09T10:00:00+08:00  2021-03-09T10:29:00+08:00  98.000000  183.000000  135.400000
4  Tomcat02           trace.self.duration_p95                   30  2021-03-09T10:00:00+08:00  2021-03-09T10:29:00+08:00   0.035000    1.005000    0.100363
5  Tomcat02        trace.to_MG02.duration_p95                   30  2021-03-09T10:00:00+08:00  2021-03-09T10:29:00+08:00   0.543200    3.644000    1.374552
6      MG02  trace.from_Tomcat02.duration_p95                   30  2021-03-09T10:00:00+08:00  2021-03-09T10:29:00+08:00   0.543200    3.644000    1.374552)```
```

In [ ]:
# In[12]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing dataframes if present; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')

# Ensure timestamp tz-aware in Asia/Shanghai for metric and log
def ensure_tz(df, ts_col='timestamp'):
    if ts_col not in df.columns:
        return df
    col = df[ts_col]
    if pd.api.types.is_integer_dtype(col) or pd.api.types.is_float_dtype(col):
        df[ts_col] = pd.to_datetime(col, unit='s', errors='coerce').dt.tz_localize(tz)
    else:
        coerced = pd.to_datetime(col, errors='coerce')
        if coerced.dt.tz is None:
            df[ts_col] = coerced.dt.tz_localize(tz)
        else:
            df[ts_col] = coerced.dt.tz_convert(tz)
    return df

metric_df = ensure_tz(metric_df, 'timestamp')
log_df = ensure_tz(log_df, 'timestamp')

# Ensure numeric 'value'
metric_df['value'] = pd.to_numeric(metric_df.get('value'), errors='coerce')
log_df['value'] = pd.to_numeric(log_df.get('value'), errors='coerce')

# Define windows (inclusive) in UTC+8
t2_start = tz.localize(pd.Timestamp('2021-03-09 10:03:00'))
t2_end   = tz.localize(pd.Timestamp('2021-03-09 10:07:00'))
t4_start = tz.localize(pd.Timestamp('2021-03-09 10:13:00'))
t4_end   = tz.localize(pd.Timestamp('2021-03-09 10:17:00'))

# Combined mask for metric (Tomcat02 window OR Tomcat04 window)
mask_metric = (
    ((metric_df['cmdb_id'] == 'Tomcat02') & (metric_df['timestamp'] >= t2_start) & (metric_df['timestamp'] <= t2_end)) |
    ((metric_df['cmdb_id'] == 'Tomcat04') & (metric_df['timestamp'] >= t4_start) & (metric_df['timestamp'] <= t4_end))
)

metric_focused = metric_df[mask_metric].copy()

# 1) Aggregate metric by (cmdb_id, kpi_name)
if metric_focused.empty:
    focused_metric_support_df = pd.DataFrame(columns=['cmdb_id','kpi_name','count','min_value','max_value','mean_value'])
else:
    focused_metric_support_df = (
        metric_focused
        .groupby(['cmdb_id','kpi_name'], dropna=False)['value']
        .agg(count='count', min_value='min', max_value='max', mean_value='mean')
        .reset_index()
    )
    # Round numeric columns for compactness
    focused_metric_support_df['min_value'] = focused_metric_support_df['min_value'].round(6)
    focused_metric_support_df['max_value'] = focused_metric_support_df['max_value'].round(6)
    focused_metric_support_df['mean_value'] = focused_metric_support_df['mean_value'].round(6)
    focused_metric_support_df = focused_metric_support_df.sort_values('count', ascending=False).reset_index(drop=True)

# Prepare compact top-20 for display
focused_metric_support_df_head20 = focused_metric_support_df.head(20)

# 2) Filter logs for the same components/windows and select requested columns
mask_log = (
    ((log_df['cmdb_id'] == 'Tomcat02') & (log_df['timestamp'] >= t2_start) & (log_df['timestamp'] <= t2_end)) |
    ((log_df['cmdb_id'] == 'Tomcat04') & (log_df['timestamp'] >= t4_start) & (log_df['timestamp'] <= t4_end))
)

focused_log_support_df = log_df[mask_log].copy()
# Select columns and sort by timestamp asc
cols_needed = [c for c in ['timestamp','cmdb_id','log_name','value'] if c in focused_log_support_df.columns]
if not cols_needed:
    focused_log_support_df = pd.DataFrame(columns=['timestamp','cmdb_id','log_name','value'])
else:
    focused_log_support_df = focused_log_support_df[cols_needed].sort_values('timestamp', ascending=True).reset_index(drop=True)

# Compact display: up to 50 rows
focused_log_support_df_head50 = focused_log_support_df.head(50)

# Display the two compact tables (full aggregated/stored tables are in focused_metric_support_df and focused_log_support_df)
focused_metric_support_df_head20, focused_log_support_df_head50

```
Out[12]:
```
Summary of focused support evidence for the two spike windows:

Windows examined
- Tomcat02: 2021-03-09 10:03:00 → 10:07:00 (UTC+8)
- Tomcat04: 2021-03-09 10:13:00 → 10:17:00 (UTC+8)

Metric aggregates (top ~20 groups shown; each group covers 5 minutes in the requested window)
- Many Tomcat session-related KPIs are present but zero-valued across the window (several Tomcat-Sessions_* KPIs for Tomcat02 and Tomcat04: min=0.0, max=0.0, mean=0.0).
- Tomcat02 notable non-zero metrics:
  - Tomcat-Threads_"http-nio-8003"_CurrentThreads: count=5, min=max=mean=232.0 (steady thread count ~232).
  - Tomcat-Threads_"http-nio-8003"_MaxThreads: count=5, value=500.0.
  - Tomcat-Requests_"http-nio-8003"_ProcessingTime (and related RequestCount/ErrorCount): large numeric values in this window (e.g., ProcessingTime min≈190,472,833 and max≈190,943,989; RequestCount values ~194,100–195,180; ErrorCount=231). These large-processing values deserve review (units/normalization).
- Tomcat04 metric groups in this narrow window are mostly session KPIs with zero values (min/max/mean = 0.0 in the aggregated result shown).

Log rows (25 rows returned across both windows; ordered by timestamp)
- Tomcat02 (10:03–10:07 window) examples:
  - 10:04:00 — log.gc_pause_max = 0.10 ; log.row_count = 1.00 ; log.error_count = 0.00
  - 10:05:00 — log.gc_pause_max = 0.09 ; log.row_count = 1.00 ; log.error_count = 0.00
  - These show small GC pause values and essentially no log errors at the Tomcat02 spike time (10:05).
- Tomcat04 (10:13–10:17 window) examples:
  - 10:16:00 — log.row_count = 347.00 ; log.error_count = 0.00
  - 10:17:00 — log.row_count = 330.00 ; log.gc_pause_max = 0.03 ; log.error_count = 0.00
  - These show a clear increase in log.row_count around 10:16 for Tomcat04, with no log.error_count increases.

Interpretation and quick recommendations
- Tomcat02 spike at ~10:05:
  - Trace spike (previously observed) at ~10:05 corresponds with elevated trace duration (self.duration_p95 and to_MG02.duration_p95 spike).
  - Metric evidence: thread count steady (~232) and extremely large request-processing numbers present in metric aggregates for Tomcat02 during the window. Log evidence: small GC pauses (~0.09–0.10s) and no log.error_count increases.
  - Suggestion: inspect the large Request ProcessingTime/RequestCount metric values (confirm units/normalization), and correlate per-request traces and application logs for 10:04–10:06 to see which requests caused long durations.
- Tomcat04 spike at ~10:15–10:16:
  - Trace spike at 10:15 (self.duration_p95) and row_count peak at 10:16 (trace.to_MG02.row_count = 183).
  - Metric aggregates show session KPIs zero in the narrow window (no obvious metric increase), but logs show a clear increase in log.row_count at 10:16 for Tomcat04 (347 → 330), with no log.error_count rise.
  - Suggestion: focus on request-rate / traffic surge at ~10:16 (Tomcat04 → MG02). Examine request-level traces and any upstream load changes. Check application logs around 10:15–10:17 for request patterns (the stored log rows include high row_count at 10:16).

Stored kernel variables
- Aggregated metric support table: focused_metric_support_df
- Log rows for both windows: focused_log_support_df
- (You previously also have extracted trace series variables for the groups of interest.)

If you want, I can:
- Extract raw request/trace IDs and nearby log messages for the specific spike timestamps (e.g., Tomcat02 at 2021-03-09 10:05, Tomcat04 at 2021-03-09 10:15–10:16), or
- Show the full per-minute metric rows for the specific suspicious KPI names during these windows. Which do you prefer?

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  count    min_value    max_value   mean_value
0   Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...      5          0.0          0.0          0.0
1   Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...      5          0.0          0.0          0.0
2   Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...      5        232.0        232.0        232.0
3   Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...      5          0.0          4.0          1.8
4   Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...      5        500.0        500.0        500.0
5   Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...      5          0.0          0.0          0.0
6   Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...      5          0.0          0.0          0.0
7   Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...      5          0.0          0.0          0.0
8   Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...      5          0.0          0.0          0.0
9   Tomcat02     Tomcat-Sessions_7441--_SESSIONRejectedSessions      5          0.0          0.0          0.0
10  Tomcat02     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter      5          0.0          0.0          0.0
11  Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...      5          0.0          0.0          0.0
12  Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...      5          0.0          0.0          0.0
13  Tomcat02  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...      5          0.0          0.0          0.0
14  Tomcat02        Tomcat-Sessions_7441--_SESSIONActiveCounter      5          0.0          0.0          0.0
15  Tomcat02  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...      5          0.0          0.0          0.0
16  Tomcat02    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter      5          0.0          0.0          0.0
17  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_Processin...      5  190472833.0  190943989.0  190701754.4
18  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...      5     194100.0     195180.0     194652.6
19  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...      5        231.0        231.0        231.0,                    timestamp   cmdb_id          log_name   value
0  2021-03-09 10:04:00+08:00  Tomcat02   log.error_count    0.00
1  2021-03-09 10:04:00+08:00  Tomcat02  log.gc_pause_max    0.10
2  2021-03-09 10:04:00+08:00  Tomcat02     log.row_count    1.00
3  2021-03-09 10:05:00+08:00  Tomcat02   log.error_count    0.00
4  2021-03-09 10:05:00+08:00  Tomcat02  log.gc_pause_max    0.09
..                       ...       ...               ...     ...
20 2021-03-09 10:16:00+08:00  Tomcat04   log.error_count    0.00
21 2021-03-09 10:16:00+08:00  Tomcat04     log.row_count  347.00
22 2021-03-09 10:17:00+08:00  Tomcat04  log.gc_pause_max    0.03
23 2021-03-09 10:17:00+08:00  Tomcat04   log.error_count    0.00
24 2021-03-09 10:17:00+08:00  Tomcat04     log.row_count  330.00

[25 rows x 4 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[13]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing aggregated anomaly tables and extracted series if available
trace_anom = globals().get('trace_anomalies_df', pd.DataFrame())
trace_low_anom = globals().get('trace_low_anomalies_df', pd.DataFrame())
metric_anom = globals().get('metric_anomalies_df', pd.DataFrame())
log_anom = globals().get('log_anomalies_df', pd.DataFrame())

# Also reuse focused support and extracted series
focused_metrics = globals().get('focused_metric_support_df', pd.DataFrame())
focused_logs = globals().get('focused_log_support_df', pd.DataFrame())

# Extracted per-series (may exist)
t2_self_p95 = globals().get('tomcat02_self_duration_p95_series', pd.DataFrame())
t2_to_mg02_p95 = globals().get('tomcat02_to_MG02_duration_p95_series', pd.DataFrame())
t4_self_p95 = globals().get('tomcat04_self_duration_p95_series', pd.DataFrame())
t4_to_mg02_rc = globals().get('tomcat04_to_MG02_rowcount_series', pd.DataFrame())
mg02_from_t4 = globals().get('mg02_from_tomcat04_rowcount_series', pd.DataFrame())
mg02_from_t2 = globals().get('mg02_from_tomcat02_duration_p95_series', pd.DataFrame())

# Candidate selection based on observed anomalies earlier: Tomcat02, Tomcat04, MG02
candidates = [
    {'component':'Tomcat02', 'trace_keys':['trace.self.duration_p95','trace.to_MG02.duration_p95']},
    {'component':'Tomcat04', 'trace_keys':['trace.self.duration_p95','trace.to_MG02.row_count']},
    {'component':'MG02',     'trace_keys':['trace.from_Tomcat02.duration_p95','trace.from_Tomcat04.row_count']}
]

rc_rows = []
rc_support = {}

def fmt_ts(x):
    if pd.isna(x):
        return None
    if isinstance(x, pd.Timestamp):
        return x.isoformat()
    try:
        return pd.Timestamp(x).isoformat()
    except Exception:
        return str(x)

def short(v, n=6):
    try:
        if pd.isna(v):
            return "NaN"
        fv = float(v)
        if abs(fv) >= 1e6:
            return f"{int(fv):,}"
        return f"{round(fv, n)}"
    except Exception:
        return str(v)

for cand in candidates:
    comp = cand['component']
    keys = cand['trace_keys']
    evidence = []
    # Find matching high-side trace anomalies (trace_anom)
    ta_matches = trace_anom[(trace_anom.get('cmdb_id') == comp) & (trace_anom.get('trace_name').isin(keys))] if not trace_anom.empty else pd.DataFrame()
    if not ta_matches.empty:
        # take earliest anomaly timestamp among matches
        earliest = ta_matches['earliest_anomaly_timestamp'].min()
        earliest_s = fmt_ts(earliest)
        # build evidence items per matched row (limit few)
        for _, r in ta_matches.head(4).iterrows():
            tn = r['trace_name']
            ts = fmt_ts(r['earliest_anomaly_timestamp'])
            ev = f"{tn}@{ts} max={short(r['max_value_in_window'])} p95={short(r['p95_value'])} sev={short(r['severity_ratio'])}"
            evidence.append(ev)
    else:
        # check low-tail anomalies if relevant
        tlo = trace_low_anom[(trace_low_anom.get('cmdb_id') == comp) & (trace_low_anom.get('trace_name').isin(keys))] if not trace_low_anom.empty else pd.DataFrame()
        if not tlo.empty:
            earliest = tlo['earliest_anomaly_timestamp'].min()
            earliest_s = fmt_ts(earliest)
            for _, r in tlo.head(4).iterrows():
                tn = r['trace_name']
                ts = fmt_ts(r['earliest_anomaly_timestamp'])
                ev = f"{tn}@{ts} min={short(r['min_value_in_window'])} p15={short(r['p15_value'])} sev={short(r['severity_ratio'])}"
                evidence.append(ev)
        else:
            earliest_s = None

    # Add supporting metric evidence from focused_metrics if available
    if not focused_metrics.empty:
        fm = focused_metrics[focused_metrics['cmdb_id'] == comp]
        if not fm.empty:
            # pick top 3 metric rows by count or by max_value
            fm_samp = fm.sort_values(['count','max_value'], ascending=[False, False]).head(3)
            for _, r in fm_samp.iterrows():
                ev = f"metric:{r['kpi_name']} cnt={int(r['count'])} min={short(r['min_value'])} max={short(r['max_value'])}"
                evidence.append(ev)

    # Add supporting log evidence from focused_logs if available
    if not focused_logs.empty:
        fl = focused_logs[focused_logs['cmdb_id'] == comp]
        if not fl.empty:
            # take up to 3 earliest log rows
            for _, r in fl.head(3).iterrows():
                ts = fmt_ts(r['timestamp'])
                lname = r.get('log_name','')
                ev = f"log:{lname}@{ts}={short(r.get('value'))}"
                evidence.append(ev)

    # If still no evidence, try to add extracted per-series spike examples
    series_map = {
        'Tomcat02': [t2_self_p95, t2_to_mg02_p95],
        'Tomcat04': [t4_self_p95, t4_to_mg02_rc],
        'MG02':     [mg02_from_t2, mg02_from_t4]
    }
    for s in series_map.get(comp, []):
        if isinstance(s, pd.DataFrame) and not s.empty:
            # choose max row
            try:
                row_max = s.loc[s['value'].idxmax()]
                ts = fmt_ts(row_max['timestamp'])
                ev = f"series:{row_max['trace_name']}@{ts} val={short(row_max['value'])}"
                evidence.append(ev)
            except Exception:
                pass

    # Propose reason based on evidence heuristics
    proposed_reason = None
    if any('duration_p95' in e or 'duration_mean' in e or 'duration' in e for e in evidence):
        proposed_reason = "high JVM CPU load"
    if any('row_count' in e for e in evidence):
        # traffic/throughput related; prefer CPU usage wording
        proposed_reason = "high CPU usage" if proposed_reason is None else proposed_reason
    if proposed_reason is None:
        proposed_reason = "performance degradation (unknown)"

    rc_support[comp] = evidence

    rc_rows.append({
        'candidate_component': comp,
        'occurrence_timestamp': earliest_s,
        'proposed_reason': proposed_reason,
        'supporting_evidence': ", ".join(evidence[:5])  # compact comma-separated short items
    })

# Build final DataFrame with up to 3 candidate rows
rc_candidates_df = pd.DataFrame(rc_rows).head(3)

# Store full supporting evidence lists in kernel variable
rc_candidates_support = rc_support

# Display compact result (DataFrame and support dict)
rc_candidates_df, rc_candidates_support

```
Out[13]:
```
```python
# Compose a plain-English root-cause candidate summary using kernel-stored results
rc_df = globals().get('rc_candidates_df')
rc_support = globals().get('rc_candidates_support', {})

if rc_df is None or rc_df.empty:
    summary = "No root-cause candidates found in kernel variables."
else:
    lines = ["Root-cause candidate report for 2021-03-09 10:00–10:30 (UTC+8):"]
    for i, r in rc_df.head(3).iterrows():
        comp = r['candidate_component']
        ts = r['occurrence_timestamp'] or "<unknown>"
        reason = r['proposed_reason'] or "performance issue"
        # pick up to 3 short support items from stored support dict
        support_list = rc_support.get(comp, [])
        short_support = support_list[:3]
        support_str = "; ".join(short_support) if short_support else "<no short evidence available>"
        lines.append(f"{i+1}. component: {comp}")
        lines.append(f"   - occurrence_timestamp: {ts}")
        lines.append(f"   - proposed_reason: {reason}")
        lines.append(f"   - supporting_evidence: {support_str}")
    lines.append("")
    lines.append("Notes:")
    lines.append("- Full supporting evidence lists are saved in kernel variable 'rc_candidates_support'.")
    lines.append("- These are hypotheses based on trace/metric/log anomalies in the 10:00–10:30 UTC+8 window.")
    lines.append("- No telemetry exists for 18:00–18:30, so no inference was made for that interval.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(  candidate_component       occurrence_timestamp    proposed_reason                                supporting_evidence
0            Tomcat02  2021-03-09T10:05:00+08:00  high JVM CPU load  trace.self.duration_p95@2021-03-09T10:05:00+08...
1            Tomcat04  2021-03-09T10:05:00+08:00  high JVM CPU load  trace.to_MG02.row_count@2021-03-09T10:09:00+08...
2                MG02  2021-03-09T10:05:00+08:00  high JVM CPU load  trace.from_Tomcat04.row_count@2021-03-09T10:09..., {'Tomcat02': ['trace.self.duration_p95@2021-03-09T10:05:00+08:00 max=1.005 p95=0.367767 sev=1.732708', 'trace.to_MG02.duration_p95@2021-03-09T10:05:00+08:00 max=3.644 p95=2.11458 sev=0.723274', 'metric:Tomcat-Requests_7441-"http-nio-8003"_ProcessingTimeRequestInfo cnt=5 min=190,472,833 max=190,943,989', 'metric:Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRequestInfo cnt=5 min=330054.0 max=330054.0', 'metric:Tomcat-Requests_7441-"http-nio-8003"_RequestCountRequestInfo cnt=5 min=194100.0 max=195180.0', 'log:log.error_count@2021-03-09T10:04:00+08:00=0.0', 'log:log.gc_pause_max@2021-03-09T10:04:00+08:00=0.1', 'log:log.row_count@2021-03-09T10:04:00+08:00=1.0', 'series:trace.self.duration_p95@2021-03-09T10:05:00+08:00 val=1.005', 'series:trace.to_MG02.duration_p95@2021-03-09T10:05:00+08:00 val=3.644'], 'Tomcat04': ['trace.to_MG02.row_count@2021-03-09T10:09:00+08:00 max=183.0 p95=174.0 sev=0.051724', 'trace.self.duration_p95@2021-03-09T10:05:00+08:00 max=1.004 p95=0.126042 sev=6.965599', 'metric:Tomcat-Requests_7441-"http-nio-8003"_ProcessingTimeRequestInfo cnt=5 min=177,151,451 max=177,735,308', 'metric:Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRequestInfo cnt=5 min=324852.0 max=324852.0', 'metric:Tomcat-Requests_7441-"http-nio-8003"_RequestCountRequestInfo cnt=5 min=194514.0 max=195762.0', 'log:log.row_count@2021-03-09T10:13:00+08:00=312.0', 'log:log.error_count@2021-03-09T10:13:00+08:00=0.0', 'log:log.gc_pause_max@2021-03-09T10:13:00+08:00=0.06', 'series:trace.self.duration_p95@2021-03-09T10:15:00+08:00 val=1.004', 'series:trace.to_MG02.row_count@2021-03-09T10:16:00+08:00 val=183.0'], 'MG02': ['trace.from_Tomcat04.row_count@2021-03-09T10:09:00+08:00 max=183.0 p95=174.0 sev=0.051724', 'trace.from_Tomcat02.duration_p95@2021-03-09T10:05:00+08:00 max=3.644 p95=2.11458 sev=0.723274', 'series:trace.from_Tomcat02.duration_p95@2021-03-09T10:05:00+08:00 val=3.644', 'series:trace.from_Tomcat04.row_count@2021-03-09T10:16:00+08:00 val=183.0']})```
```